# Rajput EL-to-IV Pipeline — Joji's working copy

Adapted from Zubair's `rajput_implement.py`. Edits made for Joji's machine
are marked with `# EDIT:` comments — search for that string to find every
place you need to check or fill in before running.

**Before running, do these in order:**
1. Fix `source_dir` (Step 0 cell below) to point at your dataset folder.
2. Either get `Pickle.py` / `Matplotlib_stylesheet.py` from Zubair/supervisor
   and drop them in a `support_scripts` folder, OR leave the fallback below
   active (it skips those imports safely if the files aren't found).
3. Run the "bit-depth check" cell early — confirms whether your TIFFs are
   16-bit, which decides whether the image-loading fix (Step 4 below) is
   needed.
4. Ngspice is already wired up for your `C:\ngspice_dll` install — no edit
   needed there unless you move that folder.

In [1]:
'''
Rajput EL-to-IV parameter extraction — refactored into functions.
Reference: Rajput et al., Sol. Energy 2018, DOI: 10.1016/j.solener.2018.07.046
Applied to the Sandia PV-IV-EL dataset.

Each pipeline step has a corresponding validate_* function that reproduces
the inline diagnostic plots/prints from rajput_implement.py.

Original: Zubair Abdullah-Vetter
Edited for local run: Joji (Ghozy Abror)
'''

import os
import sys
from datetime import datetime

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.constants import Boltzmann as k_B
from scipy.constants import elementary_charge as q_e
from scipy.constants import zero_Celsius as T_0
from scipy.interpolate import interp1d
from scipy.optimize import least_squares
from skimage import morphology

from pvlib.ivtools.sde import fit_sandia_simple
from pvlib.pvsystem import i_from_v

# directories
cwd = os.getcwd()

# EDIT: this pulled in two helper modules (Pickle.py, Matplotlib_stylesheet.py)
# from Zubair's machine. Neither is called anywhere else in this script as
# far as I can tell, so this is wrapped so a missing folder won't crash the
# whole notebook. If you get the real files from your supervisor, put them
# in a `support_scripts` folder at the path below and this will pick them up
# automatically. If some later cell errors with a NameError for something
# these files defined, that's the sign you actually need them.
libs_path = os.path.abspath(os.path.join(cwd, '..', '..', 'support_scripts'))
sys.path.append(libs_path)
try:
    from Pickle import *
    from Matplotlib_stylesheet import *
    print(f"Loaded helper modules from: {libs_path}")
except ImportError:
    print(f"Note: Pickle.py / Matplotlib_stylesheet.py not found at {libs_path} "
          f"— skipping (not required unless a later cell errors asking for them).")

# EDIT: point this at YOUR dataset root, not Zubair's OneDrive path.
# Fill in the actual folder name(s) below — this is currently a guess based
# on your working-directory convention from other chats. Update as needed.
source_dir = r"C:\Users\Ghozy Abror\OneDrive - Institut Teknologi Bandung\Karirku\UNSW\Thesis\Coding"  # EDIT THIS PATH
cell_dir   = os.path.join(source_dir, "EL_cell_low_res_2")

df = pd.read_excel(os.path.join(source_dir, "AnonDB.xlsx"),
                   sheet_name="AnonDB", usecols="A:BO", header=0, nrows=617)
df["IVPath"]  = df["IVPath"].str.replace("./IV/", "")
df["ELLPath"] = df["ELLPath"].str.replace("./EL/", "")
df["ELHPath"] = df["ELHPath"].str.replace("./EL/", "")

V_th = k_B * (25 + T_0) / q_e  # thermal voltage at 25 degC

Note: Pickle.py / Matplotlib_stylesheet.py not found at c:\Users\Ghozy Abror\OneDrive - Institut Teknologi Bandung\Karirku\UNSW\support_scripts — skipping (not required unless a later cell errors asking for them).


In [2]:
# This must run before any PySpice/ngspice simulation call (i.e. before
# sim_module_oneD() is ever used). Without it you'll hit the same
# "NGSPICE_PATH TypeError" / "cannot load library" errors we debugged earlier.
#
# If you ever move the C:\ngspice_dll folder, update the two paths below to
# match. LIBRARY_PATH must keep the literal `{}` right before `.dll` —
# PySpice fills that placeholder in itself, don't replace it.

#aku ubah euy

try:
    from PySpice.Spice.NgSpice.Shared import NgSpiceShared
    #NgSpiceShared.LIBRARY_PATH = r"C:\ngspice_dll\Spice64_dll\dll-vs\ngspice{}.dll"
    #NgSpiceShared.NGSPICE_PATH = r"C:\ngspice_dll\Spice64_dll"
    #_ng = NgSpiceShared.new_instance()
    #print("ngspice loaded OK:", _ng.exec_command('version').splitlines()[0]
    #      if _ng.exec_command('version') else "(no version string returned)")
except Exception as e:
    print(f"WARNING: ngspice did not load cleanly ({e}). "
          f"PySpice simulation cells later in this notebook will fail until "
          f"this is fixed.")

## Bit-depth check  (EDIT: new cell, not in original script)
Run this once on a real TIFF from your dataset. If `dtype` prints
`uint16` and `max` is well above 255, your source images are genuinely
16-bit — which means the image loading in `load_module_data` below
(currently using `cv2.IMREAD_GRAYSCALE`, an 8-bit read) is discarding
real precision before the ×257 rescale ever runs. See the EDIT note in
that function for the fix, and flag this to your supervisor before
changing it, since it's a deviation from Zubair's original code.

In [3]:
# EDIT: point this at any single real TIFF from your dataset to check bit depth
_sample_tiff_path = None  # e.g. r"C:\...\EL_cell_low_res\<module>_80\<module>_80_001.tiff"
if _sample_tiff_path:
    _sample = cv2.imread(_sample_tiff_path, cv2.IMREAD_UNCHANGED)
    print("dtype:", _sample.dtype, "| max value:", _sample.max())
    if _sample.dtype == np.uint8:
        print("-> Source is genuinely 8-bit. Current IMREAD_GRAYSCALE loading is fine.")
    else:
        print("-> Source is >8-bit. Current IMREAD_GRAYSCALE loading is DOWNCONVERTING "
              "before the rescale — see EDIT note in load_module_data().")
else:
    print("Set _sample_tiff_path above to run this check.")

Set _sample_tiff_path above to run this check.


In [4]:

def quick_plot(img, title=None, cmap="inferno", cbar=False):
    """Display a single image with optional colourbar."""
    plt.imshow(img, cmap=cmap)
    plt.title(title)
    plt.xticks([]); plt.yticks([])
    if cbar:
        plt.colorbar(shrink=0.5)
    plt.show()


def plot_module_images(module_imgs, nrows=10, ncols=6, cmap="inferno",
                       cbar=False, title=None, cbar_title=None, save_path=None):
    """Plot a nrows x ncols grid of cell images with shared colour scale."""
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 2, nrows * 2),
                             gridspec_kw=dict(wspace=0.03, hspace=0.03))
    vmin = np.nanpercentile(module_imgs, 1)
    vmax = np.nanpercentile(module_imgs, 99)
    for idx, ax in enumerate(axes.flat):
        ax.imshow(module_imgs[idx], cmap=cmap, vmin=vmin, vmax=vmax)
        ax.axis("off")
    if cbar:
        cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
        norm = plt.Normalize(vmin=vmin, vmax=vmax)
        sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
        sm.set_array([])
        fig.colorbar(sm, cax=cbar_ax)
        cbar_ax.yaxis.set_tick_params(labelsize=30)
        cbar_ax.yaxis.set_label_position('right')
        cbar_ax.yaxis.set_label_text(cbar_title, fontsize=40)
    if title:
        plt.suptitle(title, fontsize=16)
    if save_path:
        plt.savefig(save_path, dpi=150)
    plt.show()
    plt.close()

In [5]:

def _center_square_mask(h, w, ratio):
    """Return a filled square mask centred on (H, W); side = ratio * min(h, w)."""
    mask = np.zeros((h, w), dtype=np.uint8)
    side = int(min(h, w) * ratio)
    tl = (w // 2 - side // 2, h // 2 - side // 2)
    br = (w // 2 + side // 2, h // 2 + side // 2)
    cv2.rectangle(mask, tl, br, color=1, thickness=-1)
    return mask.astype(np.float32)


def create_busbar_mask(cell, IBC=False, busbar_orientation="horizontal"):
    """
    Build a boolean active-area mask for a cell image.

    - For standard cells (IBC=False): removes busbars detected as intensity
      troughs in the averaged line profile, plus dark-edge pixels.
    - For IBC cells (IBC=True): uses only the percentile and centre-fill masks
      (no busbars to suppress).

    Returns bool array (True = active silicon).
    """
    H, W = cell.shape
    dim = H if busbar_orientation == "horizontal" else W

    # build average line profile perpendicular to busbar direction
    profiles = [cell[:, i] if busbar_orientation == "horizontal" else cell[i, :]
                for i in range(dim)]
    avg_profile = np.sum(profiles, axis=0)
    threshold   = np.percentile(avg_profile, 10)
    bb_idx      = np.where(avg_profile < threshold)[0]

    busbar_mask = np.zeros_like(cell, dtype=bool)
    if busbar_orientation == "horizontal":
        busbar_mask[bb_idx, :] = True
    else:
        busbar_mask[:, bb_idx] = True
    busbar_mask = ~busbar_mask  # True = NOT a busbar

    # percentile edge mask
    mask = cell > np.nanpercentile(cell, 1)

    # centre-fill to keep inner defects inside masked area
    center_mask = _center_square_mask(H, W, 0.9).astype(bool)
    center_mask = center_mask & (cell > 0)
    mask = mask | center_mask

    if not IBC:
        mask = mask & busbar_mask

    return mask


def validate_busbar_masks(cell_masks, cells_lo, cells_hi, nrows, ncols,
                          sample_idx=47):
    """
    Visualise busbar masks: module grid + one example overlap image.
    """
    masks_grid = [(m.astype(np.uint8) * 255) for m in cell_masks]
    plot_module_images(masks_grid, nrows=nrows, ncols=ncols, cmap="gray",
                       cbar=True, cbar_title="Mask (0/1)",
                       title="Busbar masks — all cells")

    i = sample_idx
    cell_img = cells_hi[i]
    mask     = cell_masks[i]
    overlap  = np.zeros_like(cell_img, dtype=np.float32)
    overlap[mask] = cell_img[mask]
    quick_plot(cell_masks[i].astype(np.uint8), title=f"Cell {i+1} mask", cmap="gray", cbar=True)
    quick_plot(cell_img,  title=f"Cell {i+1} hi-bias EL", cmap="inferno", cbar=True)
    quick_plot(overlap,   title=f"Cell {i+1} masked EL",  cmap="inferno", cbar=True)

In [6]:

def load_module_data(mod_name, IBC, cell_dir, df):
    """
    Load and preprocess cell images + metadata for one module.
    """

    mod_name = str(mod_name).strip()

    # ------------------------------------------------------------
    # 1. Cell image folders
    # ------------------------------------------------------------
    mod_hi_path = os.path.join(cell_dir, f"{mod_name}_80")
    mod_lo_path = os.path.join(cell_dir, f"{mod_name}_20")

    if not os.path.isdir(mod_hi_path):
        raise FileNotFoundError(f"Missing high-bias folder: {mod_hi_path}")

    if not os.path.isdir(mod_lo_path):
        raise FileNotFoundError(f"Missing low-bias folder: {mod_lo_path}")

    # Keep False to reproduce the original 8-bit loading behaviour.
    USE_UNCHANGED_READ = False

    read_flag = (
        cv2.IMREAD_UNCHANGED
        if USE_UNCHANGED_READ
        else cv2.IMREAD_GRAYSCALE
    )

    valid_extensions = (".tif", ".tiff")

    hi_files = sorted(
        f for f in os.listdir(mod_hi_path)
        if f.lower().endswith(valid_extensions)
    )

    lo_files = sorted(
        f for f in os.listdir(mod_lo_path)
        if f.lower().endswith(valid_extensions)
    )

    cells_hi = [
        cv2.imread(os.path.join(mod_hi_path, f), read_flag)
        for f in hi_files
    ]

    cells_lo = [
        cv2.imread(os.path.join(mod_lo_path, f), read_flag)
        for f in lo_files
    ]

    # Check failed image reads
    bad_hi = [f for f, img in zip(hi_files, cells_hi) if img is None]
    bad_lo = [f for f, img in zip(lo_files, cells_lo) if img is None]

    if bad_hi:
        raise ValueError(
            f"Failed to read high-bias images for {mod_name}: {bad_hi[:5]}"
        )

    if bad_lo:
        raise ValueError(
            f"Failed to read low-bias images for {mod_name}: {bad_lo[:5]}"
        )

    if len(cells_hi) == 0:
        raise ValueError(f"No high-bias TIFF cells found for {mod_name}")

    if len(cells_lo) == 0:
        raise ValueError(f"No low-bias TIFF cells found for {mod_name}")

    if len(cells_hi) != len(cells_lo):
        raise ValueError(
            f"Different cell counts for {mod_name}: "
            f"high={len(cells_hi)}, low={len(cells_lo)}"
        )

    num_cells = len(cells_hi)

    if num_cells == 60:
        nrows, ncols = 6, 10
    elif num_cells == 72:
        nrows, ncols = 6, 12
    elif num_cells % 6 == 0:
        nrows, ncols = 6, num_cells // 6
    else:
        raise ValueError(
            f"Unsupported number of cells for {mod_name}: {num_cells}"
        )

    # ------------------------------------------------------------
    # 2. Metadata lookup
    # ------------------------------------------------------------
    if "module" in df.columns:
        module_values = (
            df["module"]
            .astype(str)
            .str.strip()
        )

        mod_row = df.loc[module_values == mod_name].copy()

    elif "mod_name" in df.columns:
        module_values = (
            df["mod_name"]
            .astype(str)
            .str.strip()
        )

        mod_row = df.loc[module_values == mod_name].copy()

    else:
        # Fallback: recreate module name from ELHPath
        if "ELHPath" not in df.columns:
            raise KeyError(
                "Metadata requires one of these columns: "
                "'module', 'mod_name', or 'ELHPath'."
            )

        module_values = (
            df["ELHPath"]
            .astype(str)
            .str.replace("\\", "/", regex=False)
            .str.rsplit("/", n=1)
            .str[-1]
            .str.replace(r"\.tiff?$", "", regex=True, case=False)
            .str.replace(r"_80$", "", regex=True)
            .str.strip()
        )

        mod_row = df.loc[module_values == mod_name].copy()

    if mod_row.empty:
        raise ValueError(f"Metadata lookup failed for {mod_name}")

    if len(mod_row) > 1:
        print(
            f"Warning: {len(mod_row)} metadata rows found for "
            f"{mod_name}; using the first row."
        )
        mod_row = mod_row.iloc[[0]].copy()

    # ------------------------------------------------------------
    # 3. Busbar masks
    # ------------------------------------------------------------
    cell_masks = [
        create_busbar_mask(
            cell,
            IBC=IBC,
            busbar_orientation="horizontal",
        )
        for cell in cells_lo
    ]

    # ------------------------------------------------------------
    # 4. Convert image values to float32
    # ------------------------------------------------------------
    if not USE_UNCHANGED_READ:
        cells_hi = [
            cell.astype(np.float32) * (65535.0 / 255.0)
            for cell in cells_hi
        ]

        cells_lo = [
            cell.astype(np.float32) * (65535.0 / 255.0)
            for cell in cells_lo
        ]
    else:
        cells_hi = [
            cell.astype(np.float32)
            for cell in cells_hi
        ]

        cells_lo = [
            cell.astype(np.float32)
            for cell in cells_lo
        ]

    # ------------------------------------------------------------
    # 5. Prevent log(0)
    # ------------------------------------------------------------
    cells_hi = [
        np.where(mask & (cell == 0), 1, cell)
        for cell, mask in zip(cells_hi, cell_masks)
    ]

    cells_lo = [
        np.where(mask & (cell == 0), 1, cell)
        for cell, mask in zip(cells_lo, cell_masks)
    ]

    # ------------------------------------------------------------
    # 6. Read and validate metadata values
    # ------------------------------------------------------------
    hi_exp = float(
        mod_row["High_Sensor_Exposure_Time_(s)"].iloc[0]
    )

    lo_exp = float(
        mod_row["Low_Sensor_Exposure_Time_(s)"].iloc[0]
    )

    if not np.isfinite(hi_exp) or hi_exp <= 0:
        raise ValueError(
            f"Invalid high exposure time for {mod_name}: {hi_exp}"
        )

    if not np.isfinite(lo_exp) or lo_exp <= 0:
        raise ValueError(
            f"Invalid low exposure time for {mod_name}: {lo_exp}"
        )

    # Normalise by exposure time
    cells_hi = [cell / hi_exp for cell in cells_hi]
    cells_lo = [cell / lo_exp for cell in cells_lo]

    return dict(
        cells_hi=cells_hi,
        cells_lo=cells_lo,
        cell_masks=cell_masks,
        mod_row=mod_row,
        nrows=nrows,
        ncols=ncols,
        num_cells=num_cells,

        lo_applied_I=float(
            mod_row["Low_Applied_Current_(A)"].iloc[0]
        ),
        lo_applied_V=float(
            mod_row["Low_Applied_Voltage_(V)"].iloc[0]
        ),
        hi_applied_I=float(
            mod_row["High_Applied_Current_(A)"].iloc[0]
        ),
        hi_applied_V=float(
            mod_row["High_Applied_Voltage_(V)"].iloc[0]
        ),

        hi_exposure_time=hi_exp,
        lo_exposure_time=lo_exp,
    )


def validate_loaded_data(data):
    """Print exposure/applied conditions and display hi/lo module image grids."""
    d = data
    print(f"Hi-bias exposure: {d['hi_exposure_time']} s | "
          f"Lo-bias exposure: {d['lo_exposure_time']} s")
    print(f"Lo-bias:  I = {d['lo_applied_I']:.2f} A  V = {d['lo_applied_V']:.2f} V")
    print(f"Hi-bias:  I = {d['hi_applied_I']:.2f} A  V = {d['hi_applied_V']:.2f} V")
    plot_module_images(np.array(d['cells_hi']), nrows=d['nrows'], ncols=d['ncols'],
                       cmap="inferno", cbar=True, cbar_title="Pixel Intensity (a.u.)",
                       title="Hi-bias EL images")
    plot_module_images(np.array(d['cells_lo']), nrows=d['nrows'], ncols=d['ncols'],
                       cmap="inferno", cbar=True, cbar_title="Pixel Intensity (a.u.)",
                       title="Lo-bias EL images")

In [7]:

def calculate_constant_f(cell_images, cell_masks, current_I, terminal_voltage_VT,
                          T_celsius, N=60):
    """
    Extract the constant factor f (Eq. 16).
    f accounts for the proportionality between EL intensity and local recombination current.

    Args:
        cell_images: list of 2-D float arrays (EL intensities, one per cell).
        cell_masks:  list of bool masks (True = active pixel).
        current_I:   applied current at low bias [A].
        terminal_voltage_VT: measured module terminal voltage at low bias [V].
        T_celsius:   cell temperature [degC].
        N:           total number of series cells.

    Returns:
        f (float)
    """
    U_th = k_B * (T_celsius + T_0) / q_e

    total = sum(
        (U_th / 2.0) * np.log(current_I / np.sum(1.0 / phi[mask]))
        for phi, mask in zip(cell_images, cell_masks)
    )

    f = np.exp((total - terminal_voltage_VT) / ((N / 2.0) * U_th))
    return f


def validate_constant_f(factor_f):
    """Print the extracted f value."""
    print(f"Extracted factor f = {factor_f:.4e}")

In [8]:

def calculate_cell_voltage_Vi(phi_r, mask, current_I, factor_f, T_celsius):
    """
    Calculate low-bias terminal voltage Vi for a single cell (Eq. 13).

    Vi = (U_th/2) * ln( I / (f * integral(1/Phi(r) d2r)) )

    Returns:
        Vi (float) [V]
    """
    U_th = k_B * (T_celsius + T_0) / q_e
    integral = np.sum(1.0 / phi_r[mask])
    return (U_th / 2.0) * np.log(current_I / (factor_f * integral))


def validate_cell_voltage_Vi(Vi_values, lo_applied_V, cells_lo, nrows, ncols):
    """
    Check sum(Vi) ~= terminal voltage and plot Vi heatmap + lo-bias image grid.
    """
    sum_Vi = np.sum(Vi_values)
    print(f"Sum Vi = {sum_Vi:.3f} V  |  Measured V_T = {lo_applied_V:.3f} V  "
          f"|  Delta = {sum_Vi - lo_applied_V:.3f} V")

    Vi_arr = np.array(Vi_values).reshape(nrows, ncols)
    fig, ax = plt.subplots(figsize=(ncols * 2, nrows * 2))
    sns.heatmap(Vi_arr, annot=True, fmt=".3f", cmap="viridis",
                annot_kws={"size": 14}, cbar_kws={"label": "Vi (V)"}, ax=ax)
    ax.set_title("Low-bias cell voltage Vi")
    plt.show()

    plot_module_images(np.array(cells_lo), nrows=nrows, ncols=ncols,
                       cmap="inferno", cbar=True, cbar_title="Intensity (a.u.)",
                       title="Lo-bias EL (for comparison)")

In [9]:

def calculate_calibration_constant_map(phi_r, Vi, mask, T_celsius):
    """
    Compute the pixel-wise calibration constant c(r) = Phi(r) . exp(-Vi/U_th)  (Eq. 8).

    Returns:
        c_r: float64 array, NaN outside active mask.
    """
    U_th = k_B * (T_celsius + T_0) / q_e
    c_r  = np.full_like(phi_r, np.nan, dtype=np.float64)
    c_r[mask] = phi_r[mask] / np.exp(Vi / U_th)
    return c_r


def validate_calibration_map(calibration_maps, cells_lo, cell_masks, Vi_values,
                              nrows, ncols):
    """
    Plot c(r) maps; scatter of per-cell mean c_i vs EL intensity and vs Vi.
    """
    plot_module_images(np.array(calibration_maps), nrows=nrows, ncols=ncols,
                       cmap="viridis", cbar=True, title="Calibration maps c(r)",
                       cbar_title="c(r) (a.u.)")

    c_i   = np.array([np.nanmean(c[m]) for c, m in zip(calibration_maps, cell_masks)])
    intens = [np.nanmedian(img[m]) for img, m in zip(cells_lo, cell_masks)]

    fig, ax1 = plt.subplots(figsize=(12, 6))
    ax1.scatter(c_i, intens, color='blue', label='Intensity')
    ax1.set_xlabel("Mean c_i per cell")
    ax1.set_ylabel("Median pixel intensity (a.u.)", color='blue')
    ax1.tick_params(axis='y', labelcolor='blue')
    ax2 = ax1.twinx()
    ax2.scatter(c_i, Vi_values, color='orange', label='Vi')
    ax2.set_ylabel("Vi (V)", color='orange')
    ax2.tick_params(axis='y', labelcolor='orange')
    fig.suptitle("c_i vs intensity and Vi")
    plt.show()

In [10]:

def calculate_dark_saturation_current_map(mask, c_r, factor_f):
    """
    Compute pixel-wise J0(r) = f / c(r)  (Eq. 9).

    Returns:
        J0_map: masked float64 array (np.ma, NaN outside active mask).
    """
    J0 = np.full_like(c_r, np.nan, dtype=np.float64)
    J0[mask] = factor_f / c_r[mask]
    return np.ma.array(J0, mask=np.isnan(J0))


def validate_j0_maps(J0_maps, effective_J0_values, nrows, ncols):
    """
    Heatmap of per-cell effective J0, log10 maps across module, J0 histogram.
    """
    J0_arr = np.array(effective_J0_values).reshape(nrows, ncols)
    fig, ax = plt.subplots(figsize=(ncols * 2, nrows * 2))
    sns.heatmap(J0_arr, annot=True, fmt=".2e", cmap="viridis_r",
                annot_kws={"size": 14}, cbar_kws={"label": "Effective J0 (A)"}, ax=ax)
    ax.set_title("Effective J0 per cell")
    plt.show()

    cmap_j0 = plt.cm.viridis.copy()
    cmap_j0.set_bad(color='lightgray')
    plot_module_images(np.log10(np.array(J0_maps)), nrows=nrows, ncols=ncols,
                       cmap=cmap_j0, cbar=True, cbar_title="log10 J0 (A/px)",
                       title="log10 J0(r) maps")

    flat = np.log10(np.array(J0_maps).flatten())
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.hist(flat[~np.isnan(flat)], bins=40, color='steelblue', alpha=0.8)
    ax.set_xlabel("log10 J0 (A/px)"); ax.set_ylabel("Frequency")
    ax.set_title("Distribution of J0 pixel values")
    plt.show()

    print(f"Module effective J0 = {np.sum(effective_J0_values):.2e} A")

In [11]:

def calculate_high_bias_cell_voltage_Vi(phi_hi, c_r, T_celsius, mask):
    """
    Estimate high-bias terminal voltage Vi using the top 1% brightest active
    pixels as a proxy for the terminal region (Eq. 1).

    Assumption: negligible interconnect voltage drop -> slight underestimation
    of total module voltage.

    Returns:
        Vi_high (float) [V]
    """
    U_th      = k_B * (T_celsius + T_0) / q_e
    active_phi = phi_hi[mask]
    active_cr  = c_r[mask]

    p_hi = np.nanpercentile(active_phi, 99.9)
    p_lo = np.nanpercentile(active_phi, 98.9)
    idx  = np.where((active_phi <= p_hi) & (active_phi >= p_lo))

    Vi_candidates = U_th * np.log(active_phi[idx] / active_cr[idx])
    return float(np.nanmean(Vi_candidates))


def validate_high_bias_vi(Vi_hi_values, hi_applied_V, cells_hi, nrows, ncols):
    """
    Check sum(Vi_hi) ~= hi-bias terminal voltage and plot Vi heatmap + hi-bias images.
    """
    sum_Vi_hi = np.sum(Vi_hi_values)
    print(f"Sum Vi_hi = {sum_Vi_hi:.3f} V  |  Measured V_T = {hi_applied_V:.3f} V  "
          f"|  Delta = {sum_Vi_hi - hi_applied_V:.3f} V")

    Vi_hi_arr = np.array(Vi_hi_values).reshape(nrows, ncols)
    fig, ax = plt.subplots(figsize=(ncols * 2, nrows * 2))
    sns.heatmap(Vi_hi_arr, annot=True, fmt=".3f", cmap="viridis",
                annot_kws={"size": 18}, cbar_kws={"label": "Hi-bias Vi (V)"}, ax=ax)
    ax.set_title("High-bias cell voltage Vi")
    plt.show()

    plot_module_images(np.array(cells_hi), nrows=nrows, ncols=ncols,
                       cmap="inferno", cbar=True, cbar_title="Intensity (a.u.)",
                       title="Hi-bias EL (for comparison)")

In [12]:

def calculate_local_voltage_map(phi_hi, c_r, T_celsius, mask):
    """
    Compute U(r) = U_th . ln(Phi_hi(r) / c(r))  (Eq. 1 rearranged).

    Returns:
        U_r: float64 array, NaN outside active mask.
    """
    U_th  = k_B * (T_celsius + T_0) / q_e
    U_r   = np.full_like(phi_hi, np.nan, dtype=np.float64)
    U_r[mask] = U_th * np.log(phi_hi[mask] / c_r[mask])
    return U_r


def validate_local_voltage_map(U_r_maps, cell_masks, hi_applied_V, nrows, ncols):
    """
    Plot U(r) maps; heatmap of per-cell mean U(r); compare sum(meanU) ~= V_T.
    """
    cmap_u = plt.cm.inferno.copy()
    cmap_u.set_bad(color='lightgray')
    plot_module_images(np.array(U_r_maps), nrows=nrows, ncols=ncols, cmap=cmap_u,
                       cbar=True, cbar_title="U(r) (V)", title="Local voltage maps U(r)")

    avg_U = [float(np.nanmean(U[m])) for U, m in zip(U_r_maps, cell_masks)]
    print(f"Sum meanU(r) = {np.sum(avg_U):.3f} V  |  Measured V_T = {hi_applied_V:.3f} V")

    arr = np.array(avg_U).reshape(nrows, ncols)
    fig, ax = plt.subplots(figsize=(ncols * 2, nrows * 2))
    sns.heatmap(arr, annot=True, fmt=".3f", cmap="inferno",
                annot_kws={"size": 14}, cbar_kws={"label": "Mean U(r) (V)"}, ax=ax)
    ax.set_title("Mean local voltage per cell")
    plt.show()

    return avg_U

In [13]:

def calculate_local_current_density_map(J0_map, U_r_map, T_celsius, mask):
    """
    Compute J(r) = J0(r) . exp(U(r) / U_th)  (Eq. 2).

    Returns:
        J_r: float64 array, NaN outside active mask.
    """
    U_th = k_B * (T_celsius + T_0) / q_e
    J_r  = np.full_like(J0_map, np.nan, dtype=np.float64)
    J_r[mask] = J0_map[mask] * np.exp(U_r_map[mask] / U_th)
    return J_r


def validate_local_current_density_map(J_r_maps, cell_masks, hi_applied_I,
                                        nrows, ncols):
    """
    Plot J(r) maps; heatmap of integrated J per cell; compare median to I_applied.
    Returns list of integrated J values per cell.
    """
    cmap_j = plt.cm.viridis.copy()
    cmap_j.set_bad(color='lightgray')
    plot_module_images(np.array(J_r_maps), nrows=nrows, ncols=ncols, cmap=cmap_j,
                       cbar=True, cbar_title="J(r) (A/px)",
                       title="Local current density maps J(r)")

    int_J = [float(np.nansum(J[m])) for J, m in zip(J_r_maps, cell_masks)]

    arr = np.array(int_J).reshape(nrows, ncols)
    fig, ax = plt.subplots(figsize=(ncols * 2, nrows * 2))
    sns.heatmap(arr, annot=True, fmt=".2f", cmap="viridis",
                annot_kws={"size": 14}, cbar_kws={"label": "Integral J(r) (A)"}, ax=ax)
    ax.set_title("Integrated J(r) per cell")
    plt.show()

    print(f"Median Integral J(r) = {np.nanmedian(int_J):.3f} A  |  "
          f"Hi-bias applied I = {hi_applied_I:.3f} A")
    return int_J

In [14]:

def calculate_series_resistance_map(Vi_high, U_r_map, J_r_map, mask):
    """
    Compute Rs(r) = (Vi - U(r)) / J(r)  (Eq. 4).

    Returns:
        Rs_r: float64 array, NaN outside active mask or where J(r) <= 0.
    """
    Rs_r = np.full_like(U_r_map, np.nan, dtype=np.float64)
    valid = mask & (J_r_map > 0)
    Rs_r[valid] = (Vi_high - U_r_map[valid]) / J_r_map[valid]
    return Rs_r


def calculate_cell_Rs_ohm(Vi_high, U_r_map, J_r_map, mask):
    """
    Compute current-weighted effective series resistance Rs,i for a cell [Ohm].

    Rs,i = Sum(dV . J(r)) / (Sum J(r))^2   where dV = |Vi - U(r)|
    """
    valid = mask & np.isfinite(U_r_map) & np.isfinite(J_r_map) & (J_r_map > 0)
    I_i  = np.nansum(J_r_map[valid])
    dV   = np.abs(Vi_high - U_r_map[valid])
    return float(np.nansum(dV * J_r_map[valid]) / I_i ** 2) if I_i > 0 else np.nan


def validate_series_resistance_map(Rs_r_maps, effective_Rs_i_values,
                                    Vi_hi_values, hi_applied_V, hi_applied_I,
                                    nrows, ncols):
    """
    Plot Rs(r) maps; heatmap of effective Rs,i; print module Rs and interconnect estimate.
    """
    cmap_rs = plt.cm.magma.copy()
    cmap_rs.set_bad(color='lightgray')
    plot_module_images(np.array(Rs_r_maps), nrows=nrows, ncols=ncols, cmap=cmap_rs,
                       cbar=True, cbar_title="Rs(r) (Ohm.px)",
                       title="Local series resistance maps Rs(r)")

    arr = np.array(effective_Rs_i_values).reshape(nrows, ncols)
    fig, ax = plt.subplots(figsize=(ncols * 2, nrows * 2))
    sns.heatmap(arr, annot=True, fmt=".5f", cmap="magma",
                annot_kws={"size": 14}, cbar_kws={"label": "Rs,i (Ohm)"}, ax=ax)
    ax.set_title("Effective Rs per cell")
    plt.show()

    Rs_cells     = np.nansum(effective_Rs_i_values)
    Rs_intercon  = (hi_applied_V - np.sum(Vi_hi_values)) / hi_applied_I
    total_Rs     = Rs_cells + Rs_intercon
    print(f"Cell Rs         = {Rs_cells:.4f} Ohm")
    print(f"Interconnect Rs = {Rs_intercon:.4f} Ohm  ({Rs_intercon/len(effective_Rs_i_values)*1e3:.3f} mOhm per cell)")
    print(f"Total module Rs = {total_Rs:.4f} Ohm")
    return total_Rs, Rs_intercon

In [15]:

def extract_JscVoc(J, V):
    """Interpolate Isc (at V=0) and Voc (at I=0) from measured IV data."""
    Jsc = J[0] if not np.any(V < 0) else interp1d(
        V[_sign_idx(V) - 3 : _sign_idx(V) + 3],
        J[_sign_idx(V) - 3 : _sign_idx(V) + 3])(0).item()
    Voc = V[-1] if not np.any(J < 0) else interp1d(
        J[_sign_idx(J) - 3 : _sign_idx(J) + 3],
        V[_sign_idx(J) - 3 : _sign_idx(J) + 3])(0).item()
    return Jsc, Voc

def _sign_idx(arr):
    return np.where(np.diff(np.sign(arr)))[0][0]


def extract_IscVoc(I_curve, V_curve):
    """Interpolate Isc/Voc from a simulated (monotone) IV curve."""
    Isc = float(np.interp(0.0, V_curve, I_curve))
    sc  = np.where(np.diff(np.sign(I_curve)))[0]
    Voc = float(np.interp(0.0,
                          [I_curve[sc[0]], I_curve[sc[0] + 1]],
                          [V_curve[sc[0]], V_curve[sc[0] + 1]])) if len(sc) else np.nan
    return Isc, Voc


def extract_voc(I_curve, V_curve):
    """Return Voc from a simulated IV curve."""
    sc = np.where(np.diff(np.sign(I_curve)))[0]
    return float(np.interp(0.0,
                           [I_curve[sc[0]], I_curve[sc[0] + 1]],
                           [V_curve[sc[0]], V_curve[sc[0] + 1]])) if len(sc) else np.nan


def extract_pmpp(I_curve, V_curve):
    """Return (Pmpp, Vmp, Imp)."""
    P   = V_curve * I_curve
    idx = np.argmax(P)
    return P[idx], V_curve[idx], I_curve[idx]


def effective_module_j0(j0_cells):
    """Geometric mean of per-cell J0 values [A]."""
    j0 = np.asarray(j0_cells, dtype=float)
    return float(np.exp(np.mean(np.log(j0))))

In [16]:
from pvlib.pvsystem import i_from_v as pvlib_i_from_v

def fit_sdm_fixed_n(V, I, num_cells, T_celsius=25.0, n_fixed=1.0,
                    IL0=None, I00=None, Rs0=None, Rsh0=None,
                    Isc0=None, Voc0=None, method='lambertw'):
    """
    Fit a single-diode model with fixed ideality factor n.

    Fitted: IL [A], I0 [A], Rs [Ohm], Rsh [Ohm].
    Fixed:  nNsVth = n_fixed * num_cells * Vth.

    Isc0/Voc0: optional explicit values used for initial guesses when the
    IV array does not cleanly reach Isc or Voc (e.g. PySpice output).

    Returns dict with keys: IL, I0, Rs, Rsh, nNsVth, Vth, success, message, I_fit.
    """
    V = np.asarray(V, dtype=np.float64)
    I = np.asarray(I, dtype=np.float64)

    Vth         = k_B * (T_celsius + T_0) / q_e
    nNsVth_fix  = n_fixed * num_cells * Vth
    Voc_meas    = Voc0 if Voc0 is not None else float(V[-1])

    IL0  = max(Isc0 if Isc0 is not None else float(I[0]), 1e-6) if IL0 is None else IL0
    I00  = max(IL0 / (np.exp(Voc_meas / nNsVth_fix) - 1.0), 1e-15) if I00 is None else I00
    Rs0  = 0.2   if Rs0  is None else Rs0
    Rsh0 = 200.0 if Rsh0 is None else Rsh0

    x0 = np.array([np.log10(IL0), np.log10(I00), Rs0, np.log10(Rsh0)])
    lb = np.array([-6, -20,  0.0, 0.0])
    ub = np.array([ 3,  -1, 10.0, 8.0])

    def unpack(x):
        return 10**x[0], 10**x[1], x[2], 10**x[3]

    def residuals(x):
        IL, I0, Rs, Rsh = unpack(x)
        try:
            I_m = i_from_v(V, IL, I0, Rs, Rsh, nNsVth_fix, method=method)
        except Exception:
            return np.full_like(I, 1e6)
        res = I_m - I
        isc_pen = 5.0 * (I_m[np.argmin(np.abs(V))] - I[np.argmin(np.abs(V))])
        voc_pen = 5.0 * (I_m[np.argmin(np.abs(V - Voc_meas))]
                         - I[np.argmin(np.abs(V - Voc_meas))])
        return np.concatenate([res, [isc_pen, voc_pen]])

    fit = least_squares(residuals, x0, bounds=(lb, ub),
                        method='trf', loss='soft_l1', f_scale=0.05, max_nfev=5000)
    IL_f, I0_f, Rs_f, Rsh_f = unpack(fit.x)
    I_fit = i_from_v(V, IL_f, I0_f, Rs_f, Rsh_f, nNsVth_fix, method=method)

    return dict(IL=IL_f, I0=I0_f, Rs=Rs_f, Rsh=Rsh_f,
                nNsVth=nNsVth_fix, Vth=Vth, n_fixed=n_fixed,
                success=fit.success, message=fit.message, I_fit=I_fit)


def validate_iv_fit(fit_fixed_n, V, I, Isc, Voc, nNsVth, eff_I0,
                    eff_Rs_mod, Rsh, num_cells, n_fixed):
    """
    Overlay measured, diode-fit, and effective-module (PySpice SDM) IV curves.

    eff_I0 / eff_Rs_mod: effective module I0 [A] and Rs [Ohm] from PySpice SDM fit.
    """
    V_sim = np.linspace(0, Voc * 1.05, 300)

    I_diode = i_from_v(V_sim, fit_fixed_n['IL'], fit_fixed_n['I0'],
                        fit_fixed_n['Rs'], fit_fixed_n['Rsh'],
                        fit_fixed_n['nNsVth'], method='lambertw')
    I_eff   = i_from_v(V_sim, fit_fixed_n['IL'], eff_I0,
                        eff_Rs_mod, Rsh,
                        n_fixed * num_cells * k_B * (25 + T_0) / q_e,
                        method='lambertw')

    Voc_d, (Pmpp_d, _, _) = extract_voc(I_diode, V_sim), extract_pmpp(I_diode, V_sim)
    Voc_e, (Pmpp_e, _, _) = extract_voc(I_eff,   V_sim), extract_pmpp(I_eff,   V_sim)
    Pmpp_m, _, _           = extract_pmpp(I, V)

    print(f"\nDiode fit (n={n_fixed}):        I0={fit_fixed_n['I0']:.2e} A  "
          f"Rs={fit_fixed_n['Rs']:.4f} Ohm  Voc={Voc_d:.3f} V  Pmpp={Pmpp_d:.2f} W")
    print(f"Effective (PySpice SDM): I0={eff_I0:.2e} A  "
          f"Rs={eff_Rs_mod:.4f} Ohm  Voc={Voc_e:.3f} V  Pmpp={Pmpp_e:.2f} W")
    print(f"Measured:                Voc={Voc:.3f} V  Pmpp={Pmpp_m:.2f} W")

    fig, ax = plt.subplots(figsize=(20, 8))
    ax.plot(V, I, 'o', ms=4, label='Measured I-V', color='blue')
    ax.plot(V_sim, I_diode, '--', lw=4, label='Diode-fit I-V', color='green')
    ax.plot(V_sim, I_eff,   '-',  lw=2, label='Effective module (PySpice SDM)', color='red')
    ax.set_xlabel("Voltage (V)"); ax.set_ylabel("Current (A)")
    ax.set_title("Measured vs Diode-fit vs Effective module (PySpice SDM) I-V")
    ax.legend(loc='lower left')
    ax.set_xlim(0, Voc * 1.05); ax.set_ylim(0, Isc * 1.05)
    plt.subplots_adjust(right=0.50)

    row_labels = ["I0 (A)", "Rs (Ohm)", "Voc (V)", "Pmpp (W)"]
    cell_text  = [
        ["-", f"{fit_fixed_n['I0']:.2e}", f"{eff_I0:.2e}"],
        ["-", f"{fit_fixed_n['Rs']:.3f}", f"{eff_Rs_mod:.3f}"],
        [f"{Voc:.3f}",    f"{Voc_d:.2f}",   f"{Voc_e:.2f}"],
        [f"{Pmpp_m:.2f}", f"{Pmpp_d:.2f}",  f"{Pmpp_e:.2f}"],
    ]
    tbl = ax.table(cellText=cell_text, rowLabels=row_labels,
                   colLabels=["Measured", "Diode fit", "PySpice SDM"],
                   cellLoc='center', rowLoc='center', colLoc='center',
                   bbox=[0.157, 0.25, 0.42, 0.45])
    tbl.auto_set_font_size(False); tbl.set_fontsize(14)
    for (r, c), cell in tbl.get_celld().items():
        if r == 0:  cell.set_text_props(weight='bold'); cell.set_facecolor('#EAEAF2')
        if c == -1: cell.set_text_props(weight='bold'); cell.set_facecolor('#F5F5F5')
    plt.show()

    return V_sim, I_diode, I_eff

In [17]:
def fit_sdm_fixed_n(
    V,
    I,
    num_cells,
    T_celsius=25.0,
    n_fixed=1.0,
    IL0=None,
    I00=None,
    Rs0=None,
    Rsh0=None,
    Isc0=None,
    Voc0=None,
    method="lambertw",
    max_nfev=20,
):
    """
    Fit a single-diode model with fixed ideality factor n.

    Fitted:
        IL [A], I0 [A], Rs [Ohm], Rsh [Ohm]

    Fixed:
        nNsVth = n_fixed * num_cells * Vth
    """

    V = np.asarray(V, dtype=np.float64).ravel()
    I = np.asarray(I, dtype=np.float64).ravel()

    # Remove invalid IV points
    valid = np.isfinite(V) & np.isfinite(I)
    V = V[valid]
    I = I[valid]

    if len(V) < 5:
        raise ValueError(
            f"Not enough valid IV points for fitting: {len(V)}"
        )

    # Sort by voltage
    order = np.argsort(V)
    V = V[order]
    I = I[order]

    num_cells = int(num_cells)
    T_celsius = float(T_celsius)
    n_fixed = float(n_fixed)

    if num_cells <= 0:
        raise ValueError(f"Invalid num_cells: {num_cells}")

    if n_fixed <= 0:
        raise ValueError(f"Invalid n_fixed: {n_fixed}")

    Vth = k_B * (T_celsius + T_0) / q_e
    nNsVth_fix = n_fixed * num_cells * Vth

    if not np.isfinite(nNsVth_fix) or nNsVth_fix <= 0:
        raise ValueError(
            f"Invalid nNsVth: {nNsVth_fix}"
        )

    # Use point closest to V=0 as measured Isc estimate
    isc_index = int(np.argmin(np.abs(V)))
    isc_meas = float(I[isc_index])

    # Estimate Voc more safely than simply V[-1]
    if Voc0 is not None and np.isfinite(Voc0):
        Voc_meas = float(Voc0)
    else:
        voc_index = int(np.argmin(np.abs(I)))
        Voc_meas = float(V[voc_index])

    # Safe initial IL
    if IL0 is None or not np.isfinite(IL0) or IL0 <= 0:
        IL0 = Isc0 if Isc0 is not None else isc_meas

    IL0 = float(np.clip(IL0, 1e-6, 1e3))

    # Safe initial I0
    if I00 is None or not np.isfinite(I00) or I00 <= 0:
        exponent = np.clip(Voc_meas / nNsVth_fix, -100, 100)
        denominator = np.expm1(exponent)

        if not np.isfinite(denominator) or denominator <= 0:
            I00 = 1e-10
        else:
            I00 = IL0 / denominator

    I00 = float(np.clip(I00, 1e-20, 1e-1))

    # Safe initial Rs
    if Rs0 is None or not np.isfinite(Rs0):
        Rs0 = 0.2

    Rs0 = float(np.clip(Rs0, 0.0, 10.0))

    # Safe initial Rsh
    if Rsh0 is None or not np.isfinite(Rsh0) or Rsh0 <= 0:
        Rsh0 = 200.0

    Rsh0 = float(np.clip(Rsh0, 1.0, 1e8))

    x0 = np.array(
        [
            np.log10(IL0),
            np.log10(I00),
            Rs0,
            np.log10(Rsh0),
        ],
        dtype=np.float64,
    )

    lb = np.array([-6.0, -20.0, 0.0, 0.0])
    ub = np.array([3.0, -1.0, 10.0, 8.0])

    # Ensure x0 is strictly inside bounds
    eps = 1e-10
    x0 = np.clip(x0, lb + eps, ub - eps)

    print(
        "\nStarting fixed-n SDM fit\n"
        f"  points   = {len(V)}\n"
        f"  Voc      = {Voc_meas:.6g} V\n"
        f"  nNsVth   = {nNsVth_fix:.6g} V\n"
        f"  IL0      = {IL0:.6g} A\n"
        f"  I00      = {I00:.6e} A\n"
        f"  Rs0      = {Rs0:.6g} Ohm\n"
        f"  Rsh0     = {Rsh0:.6g} Ohm",
        flush=True,
    )

    def unpack(x):
        IL = 10.0 ** x[0]
        I0 = 10.0 ** x[1]
        Rs = x[2]
        Rsh = 10.0 ** x[3]
        return IL, I0, Rs, Rsh

    evaluation_count = 0

    def residuals(x):
        nonlocal evaluation_count
        evaluation_count += 1

        IL, I0, Rs, Rsh = unpack(x)

        try:
            I_model = i_from_v(
                V,
                IL,
                I0,
                Rs,
                Rsh,
                nNsVth_fix,
                method=method,
            )

            I_model = np.real_if_close(
                np.asarray(I_model),
                tol=1000,
            )

            # Reject genuinely complex output
            if np.iscomplexobj(I_model):
                return np.full(len(I) + 2, 1e6, dtype=float)

            I_model = np.asarray(I_model, dtype=np.float64)

            # Reject NaN, inf, or wrong shape
            if I_model.shape != I.shape:
                return np.full(len(I) + 2, 1e6, dtype=float)

            if not np.all(np.isfinite(I_model)):
                return np.full(len(I) + 2, 1e6, dtype=float)

            res = I_model - I

            voc_index = int(np.argmin(np.abs(V - Voc_meas)))

            isc_penalty = 5.0 * (
                I_model[isc_index] - I[isc_index]
            )
            voc_penalty = 5.0 * (
                I_model[voc_index] - I[voc_index]
            )

            output = np.concatenate(
                [res, [isc_penalty, voc_penalty]]
            )

            if not np.all(np.isfinite(output)):
                return np.full(len(I) + 2, 1e6, dtype=float)

            return output

        except Exception as exc:
            # Print only occasionally to avoid flooding output
            if evaluation_count <= 3:
                print(
                    f"i_from_v evaluation failed: {exc}",
                    flush=True,
                )

            return np.full(len(I) + 2, 1e6, dtype=float)

    fit = least_squares(
        residuals,
        x0,
        bounds=(lb, ub),
        method="trf",
        loss="soft_l1",
        f_scale=0.05,
        max_nfev=max_nfev,
        verbose=1,
    )

    IL_f, I0_f, Rs_f, Rsh_f = unpack(fit.x)

    I_fit = i_from_v(
        V,
        IL_f,
        I0_f,
        Rs_f,
        Rsh_f,
        nNsVth_fix,
        method=method,
    )

    I_fit = np.real_if_close(
        np.asarray(I_fit),
        tol=1000,
    )

    if np.iscomplexobj(I_fit):
        raise ValueError(
            "Final i_from_v result is complex."
        )

    I_fit = np.asarray(I_fit, dtype=np.float64)

    print(
        "\nFixed-n SDM fit completed\n"
        f"  success = {fit.success}\n"
        f"  message = {fit.message}\n"
        f"  nfev    = {fit.nfev}\n"
        f"  IL      = {IL_f:.6g} A\n"
        f"  I0      = {I0_f:.6e} A\n"
        f"  Rs      = {Rs_f:.6g} Ohm\n"
        f"  Rsh     = {Rsh_f:.6g} Ohm",
        flush=True,
    )

    return {
        "IL": IL_f,
        "I0": I0_f,
        "Rs": Rs_f,
        "Rsh": Rsh_f,
        "nNsVth": nNsVth_fix,
        "Vth": Vth,
        "n_fixed": n_fixed,
        "success": fit.success,
        "message": fit.message,
        "nfev": fit.nfev,
        "cost": fit.cost,
        "V_fit": V,
        "I_fit": I_fit,
    }

In [18]:
def fit_sdm_fixed_n(
    V,
    I,
    num_cells,
    T_celsius=25.0,
    n_fixed=1.0,
    IL0=None,
    I00=None,
    Rs0=None,
    Rsh0=None,
    Isc0=None,
    Voc0=None,
    method="lambertw",
    max_nfev=50,
):
    V = np.asarray(V, dtype=np.float64).ravel()
    I = np.asarray(I, dtype=np.float64).ravel()

    valid = np.isfinite(V) & np.isfinite(I)
    V = V[valid]
    I = I[valid]

    if len(V) < 5:
        raise ValueError(f"Only {len(V)} valid IV points remain.")

    order = np.argsort(V)
    V = np.ascontiguousarray(V[order], dtype=np.float64)
    I = np.ascontiguousarray(I[order], dtype=np.float64)

    Vth = k_B * (T_celsius + T_0) / q_e
    nNsVth_fix = float(n_fixed * num_cells * Vth)

    idx_isc = int(np.argmin(np.abs(V)))

    if Voc0 is not None:
        Voc_meas = float(Voc0)
    else:
        idx_zero_current = int(np.argmin(np.abs(I)))
        Voc_meas = float(V[idx_zero_current])

    IL0 = (
        max(float(Isc0 if Isc0 is not None else I[idx_isc]), 1e-6)
        if IL0 is None
        else float(IL0)
    )

    if I00 is None:
        exponent = np.clip(Voc_meas / nNsVth_fix, -100, 100)
        I00 = max(IL0 / np.expm1(exponent), 1e-15)
    else:
        I00 = float(I00)

    Rs0 = 0.2 if Rs0 is None else float(Rs0)
    Rsh0 = 200.0 if Rsh0 is None else float(Rsh0)

    # Make initial values valid
    IL0 = np.clip(IL0, 1e-6, 1e3)
    I00 = np.clip(I00, 1e-20, 1e-1)
    Rs0 = np.clip(Rs0, 0.0, 10.0)
    Rsh0 = np.clip(Rsh0, 1.0, 1e8)

    x0 = np.array(
        [
            np.log10(IL0),
            np.log10(I00),
            Rs0,
            np.log10(Rsh0),
        ],
        dtype=float,
    )

    lb = np.array([-6.0, -20.0, 0.0, 0.0])
    ub = np.array([3.0, -1.0, 10.0, 8.0])

    x0 = np.clip(x0, lb + 1e-10, ub - 1e-10)

    def unpack(x):
        return (
            10.0 ** x[0],
            10.0 ** x[1],
            x[2],
            10.0 ** x[3],
        )

    call_count = 0

    def residuals(x):
        nonlocal call_count
        call_count += 1

        IL, I0, Rs, Rsh = unpack(x)

        print(
            f"Evaluation {call_count}: "
            f"IL={IL:.6g}, I0={I0:.3e}, "
            f"Rs={Rs:.6g}, Rsh={Rsh:.6g}",
            flush=True,
        )

        print("  entering i_from_v", flush=True)

        # No try/except during diagnosis
        I_m = pvlib_i_from_v(
            voltage=V,
            photocurrent=IL,
            saturation_current=I0,
            resistance_series=Rs,
            resistance_shunt=Rsh,
            nNsVth=nNsVth_fix,
            method=method,
        )

        print("  i_from_v completed", flush=True)

        I_m = np.asarray(np.real_if_close(I_m), dtype=np.float64)

        if I_m.shape != I.shape:
            raise ValueError(
                f"Model shape {I_m.shape} differs from data shape {I.shape}."
            )

        if not np.all(np.isfinite(I_m)):
            bad = np.sum(~np.isfinite(I_m))
            print(f"  Non-finite model values: {bad}", flush=True)
            return np.full(len(I) + 2, 1e6, dtype=float)

        res = I_m - I
        idx_voc = int(np.argmin(np.abs(V - Voc_meas)))

        isc_pen = 5.0 * (I_m[idx_isc] - I[idx_isc])
        voc_pen = 5.0 * (I_m[idx_voc] - I[idx_voc])

        output = np.concatenate((res, [isc_pen, voc_pen]))

        print(
            f"  residual norm={np.linalg.norm(output):.6g}",
            flush=True,
        )

        return output

    # Test residual once before starting the optimizer
    print("\nTesting initial residual...", flush=True)
    test_residual = residuals(x0)

    print(
        f"Initial residual completed: "
        f"shape={test_residual.shape}, "
        f"finite={np.all(np.isfinite(test_residual))}",
        flush=True,
    )

    print("\nStarting least_squares...", flush=True)

    fit = least_squares(
        residuals,
        x0,
        bounds=(lb, ub),
        method="trf",
        loss="soft_l1",
        f_scale=0.05,
        max_nfev=max_nfev,
        verbose=2,
    )

    print("least_squares completed", flush=True)

    IL_f, I0_f, Rs_f, Rsh_f = unpack(fit.x)

    I_fit = pvlib_i_from_v(
        voltage=V,
        photocurrent=IL_f,
        saturation_current=I0_f,
        resistance_series=Rs_f,
        resistance_shunt=Rsh_f,
        nNsVth=nNsVth_fix,
        method=method,
    )

    return {
        "IL": IL_f,
        "I0": I0_f,
        "Rs": Rs_f,
        "Rsh": Rsh_f,
        "nNsVth": nNsVth_fix,
        "Vth": Vth,
        "n_fixed": n_fixed,
        "success": fit.success,
        "message": fit.message,
        "nfev": fit.nfev,
        "cost": fit.cost,
        "I_fit": np.asarray(I_fit, dtype=float),
    }

In [19]:
# Requires: PySpice + ngspice (set up in the "NgSpice setup" cell above)

try:
    from PySpice.Spice.Netlist import Circuit, SubCircuit
    from PySpice.Unit import *
    #from PySpice.Spice.NgSpice.Shared import NgSpiceShared
    _PYSPICE_AVAILABLE = True
except ImportError:
    _PYSPICE_AVAILABLE = False
    print("PySpice not found — skip PySpice cells.")


def build_pyspice_maps(eff_J0, eff_Rs, Rs_interconnect, fit_fn, num_cells, nrows, ncols):
    """
    Build per-cell PySpice parameter maps from Rajput-extracted values.

    The interconnect resistance is distributed equally across all cells as an
    additive offset to each cell's Rs:
        Rs_cell_i (PySpice) = Rs,i (Rajput) + Rs_interconnect / N

    Orientation: landscape (nrows x ncols) -> portrait (ncols x nrows) -> snake order.

    Returns dict with keys:
        rsmap, jmap, qmap, rshmap  — shape (portrait_rows, portrait_cols)
        portrait_rows, portrait_cols
        Rs_offset_per_cell  — the additive offset applied [Ohm]
    """
    portrait_rows = ncols
    portrait_cols = nrows

    Rs_offset = Rs_interconnect / num_cells
    eff_Rs_adj = [r + Rs_offset for r in eff_Rs]
    print(f"PySpice Rs offset per cell: {Rs_offset*1e3:.3f} mOhm  "
          f"(Rs_interconnect={Rs_interconnect:.4f} Ohm / {num_cells} cells)")

    def _to_map(values, fill_median=True):
        portrait = landscape_to_portrait_scalars(values, portrait_rows, portrait_cols)
        snake    = portrait_to_snake_order(portrait, portrait_rows, portrait_cols)
        arr      = np.array(snake, dtype=float).reshape(portrait_rows, portrait_cols)
        if fill_median:
            med = float(np.nanmedian(values))
            arr = np.where(np.isfinite(arr) & (arr > 0), arr, med)
        return arr

    rsmap  = _to_map(eff_Rs_adj)
    jmap   = _to_map(eff_J0)
    qmap   = np.full((portrait_rows, portrait_cols), fit_fn['IL'])
    rshmap = np.full((portrait_rows, portrait_cols), fit_fn['Rsh'] / num_cells)

    return dict(
        rsmap=rsmap, jmap=jmap, qmap=qmap, rshmap=rshmap,
        portrait_rows=portrait_rows, portrait_cols=portrait_cols,
        Rs_offset_per_cell=Rs_offset,
    )


def landscape_to_portrait_scalars(values, portrait_rows=10, portrait_cols=6):
    """
    Rotate a flat landscape-order scalar list to portrait row-major order.
    Landscape: portrait_cols rows x portrait_rows cols  (e.g. 6x10)
    Portrait:  portrait_rows rows x portrait_cols cols  (e.g. 10x6)
    """
    portrait = [None] * (portrait_rows * portrait_cols)
    for r_P in range(portrait_rows):
        for c_P in range(portrait_cols):
            landscape_idx = (portrait_cols - 1 - c_P) * portrait_rows + r_P
            portrait[r_P * portrait_cols + c_P] = values[landscape_idx]
    return portrait


def portrait_to_snake_order(cells, rows=10, cols=6):
    """
    Reorder portrait row-major cell list to PySpice snake-wiring order.
    Even columns: top->bottom; odd columns: bottom->top.
    """
    snake = [None] * (rows * cols)
    for r in range(rows):
        for c in range(cols):
            idx = r * cols + c
            if c % 2 == 0:
                snake[idx] = cells[idx]
            else:
                snake[c * rows + (rows - 1 - r)] = cells[idx]
    return snake


def gen_cell_one_diode_rajput(name, q=1000.0 @ u_mA, rs=10 @ u_mOhm,
                               rsh=200 @ u_Ohm, j01=1e-12, ni1=1):
    """PySpice subcircuit for a single-diode solar cell."""
    cell = SubCircuit(name, 't_out', 't_in')
    cell.model('d1', 'D', IS=j01, N=ni1, RS=0)
    cell.I(1, 't_load', 't_in', q)
    cell.R(2, 't_load', 't_out', rs)
    cell.R(3, 't_in',   't_load', rsh)
    cell.Diode(4, 't_in', 't_load', model='d1')
    return cell


class BypassDiodeRajput(SubCircuit):
    __nodes__ = ('BPD_input', 'BPD_output')
    def __init__(self, name):
        SubCircuit.__init__(self, name, *self.__nodes__)
        self.model('BypassDiodeRajput', 'D',
                   IS=680e-12, RS=0.001, N=1.003,
                   CJO=1e-12, M=0.3, EG=0.69, XTI=6)
        self.Diode(1, 'BPD_input', 'BPD_output', model='BypassDiodeRajput')


def gen_module_oneD(rows=10, cols=6, rsmap=None, qmap=None, jmap=None,
                    rshmap=None, bypass=True):
    """Build a PySpice module Circuit (1-diode per cell, row-major series string)."""
    qmap   = np.ones((rows, cols)) * 10    if qmap   is None else qmap
    j1map  = np.ones((rows, cols)) * 1e-13 if jmap   is None else jmap
    rsmap  = np.ones((rows, cols)) * 0.01  if rsmap  is None else rsmap
    rshmap = np.ones((rows, cols)) * 5     if rshmap is None else rshmap

    ckt = Circuit('module')
    for r in range(rows):
        for c in range(cols):
            nm = f'cell_{r}_{c}'
            ckt.subcircuit(gen_cell_one_diode_rajput(
                nm,
                q   = float(qmap[r, c])   * 1.0 @ u_A,
                j01 = float(j1map[r, c]),
                rs  = float(rsmap[r, c])  * 1.0 @ u_Ohm,
                rsh = float(rshmap[r, c]) * 1.0 @ u_Ohm,
            ))
            ckt.X(nm, nm, r * cols + c + 1, r * cols + c + 2)

    if bypass:
        n = rows * cols
        pts = [(1, n // 3 + 1), (n // 3 + 1, 2 * n // 3 + 1), (2 * n // 3 + 1, n + 1)]
        for i, (a, b) in enumerate(pts):
            ckt.subcircuit(BypassDiodeRajput(f'bypass_r{i+1}'))
            ckt.X(f'bypass_r{i+1}', f'bypass_r{i+1}', a, b)

    ckt.V('input', ckt.gnd, 1, 0.0)
    ckt.R('meas', rows * cols + 1, 0, 0.001 @ u_Ohm)
    return ckt


def sim_module_oneD(rows=10, cols=6, rsmap=None, qmap=None, jmap=None,
                    rshmap=None, bypass=False, return_IV=False):
    """
    Simulate module I-V via PySpice DC sweep.

    Returns (mpp, vmp, imp, voc, isc) or with I, V appended if return_IV=True.
    """
    ckt      = gen_module_oneD(rows, cols, rsmap, qmap, jmap, rshmap, bypass)
    sim      = ckt.simulator(temperature=25, nominal_temperature=25,
                             simulator='ngspice-shared')
    analysis = sim.dc(Vinput=slice(-5, rows * cols * 0.9, 0.1))

    current = np.asarray(analysis.Vinput)
    voltage = np.asarray(analysis.sweep)
    if current.size == 0:
        empty = (0.0,) * 5
        return empty + (current, voltage) if return_IV else empty

    P       = current * voltage
    idx     = np.argmax(P)
    isc, voc = extract_JscVoc(J=current, V=voltage)
    result  = (P[idx], voltage[idx], current[idx], voc, isc)
    return result + (current, voltage) if return_IV else result


def validate_pyspice_sim(V, I, Isc, Voc, V_spice, I_spice, Pmpp_meas,
                          I_rajput_pvlib, V_sim, module_effective_J0,
                          total_Rs, effective_Rs_i_values):
    """Plot and tabulate PySpice vs pvlib vs measured IV curves."""
    mpp_spice = float(np.max(V_spice * I_spice))
    isc_s, voc_s = extract_JscVoc(I_spice, V_spice)
    _, imp_s, vmp_s = 0, 0, 0  # simplified

    fig, ax = plt.subplots(figsize=(10, 8))
    ax.plot(V, I, 'o', ms=4, label='Measured', color='blue')
    ax.plot(V_sim, I_rajput_pvlib, '--', lw=2, label='Rajput (pvlib)', color='red')
    ax.plot(V_spice, I_spice, '-', lw=2, label='Rajput (PySpice)', color='darkorange')
    ax.set_xlabel("Voltage (V)"); ax.set_ylabel("Current (A)")
    ax.set_title("Measured vs Rajput I-V\n(pvlib module-level vs PySpice per-cell)")
    ax.legend(); ax.grid(True, alpha=0.3)
    ax.set_xlim(0, Voc * 1.05); ax.set_ylim(0, Isc * 1.05)
    plt.tight_layout(); plt.show()

    print(f"PySpice: Isc={isc_s:.4f} A  Voc={voc_s:.4f} V  Pmpp={mpp_spice:.2f} W")
    print(f"Measured:          Isc={Isc:.4f} A  Voc={Voc:.4f} V  Pmpp={Pmpp_meas:.2f} W")

# Ver Zubair

In [20]:

def run_rajput_pipeline(mod_name, IBC, cell_dir, df, source_dir,
                         T_celsius=25, n_fixed=1,
                         plot=True, plot_final=True, run_pyspice=False):
    """
    Run the full Rajput EL-to-IV extraction pipeline for one module.

    Steps:
        0. Load & preprocess images
        1. Factor f
        2. Low-bias cell voltage Vi
        3. Calibration map c(r)
        4. Dark saturation current J0(r)
        5. High-bias cell voltage Vi
        6. Local voltage U(r)
        7. Local current density J(r)
        8. Series resistance Rs(r)
        +  Sandia + constrained SDM IV fit

    Args:
        mod_name:    module identifier string (e.g. "2531_36_1_08062019")
        IBC:         True if IBC cell (suppresses busbar detection)
        cell_dir:    path to EL cell image directory
        df:          metadata DataFrame
        source_dir:  dataset root directory
        T_celsius:   measurement temperature [degC]
        n_fixed:     ideality factor for constrained SDM fit
        plot:        if True, produce all intermediate step diagnostic plots
        plot_final:  if True, produce the final measured vs diode-fit vs
                     PySpice SDM IV comparison plot

    Returns:
        dict with all intermediate and final results.
    """
    print(f"\n{'='*60}")
    print(f"  Module: {mod_name}  |  IBC={IBC}  |  T={T_celsius}degC")
    print(f"{'='*60}")

    # -- Step 0: load data ------------------------------------------------
    data      = load_module_data(mod_name, IBC, cell_dir, df)
    cells_hi  = data['cells_hi'];  cells_lo  = data['cells_lo']
    masks     = data['cell_masks']; mod_row   = data['mod_row']
    nrows     = data['nrows'];      ncols     = data['ncols']
    N         = data['num_cells']
    lo_I      = data['lo_applied_I']; lo_V = data['lo_applied_V']
    hi_I      = data['hi_applied_I']; hi_V = data['hi_applied_V']

    if plot:
        validate_loaded_data(data)
        validate_busbar_masks(masks, cells_lo, cells_hi, nrows, ncols)

    # -- Step 1: factor f -------------------------------------------------
    factor_f = calculate_constant_f(cells_lo, masks, lo_I, lo_V, T_celsius, N)
    if plot:
        validate_constant_f(factor_f)

    # -- Step 2: low-bias Vi ----------------------------------------------
    Vi_lo = [calculate_cell_voltage_Vi(phi, m, lo_I, factor_f, T_celsius)
             for phi, m in zip(cells_lo, masks)]
    if plot:
        validate_cell_voltage_Vi(Vi_lo, lo_V, cells_lo, nrows, ncols)

    # -- Step 3: calibration map c(r) -------------------------------------
    c_maps = [calculate_calibration_constant_map(phi, Vi, m, T_celsius)
              for phi, Vi, m in zip(cells_lo, Vi_lo, masks)]
    if plot:
        validate_calibration_map(c_maps, cells_lo, masks, Vi_lo, nrows, ncols)

    # -- Step 4: J0(r) maps -----------------------------------------------
    J0_maps    = [calculate_dark_saturation_current_map(m, c, factor_f)
                  for m, c in zip(masks, c_maps)]
    eff_J0     = [float(np.nansum(J[m])) for J, m in zip(J0_maps, masks)]
    if plot:
        validate_j0_maps(J0_maps, eff_J0, nrows, ncols)

    # -- Step 5: high-bias Vi ---------------------------------------------
    Vi_hi = [calculate_high_bias_cell_voltage_Vi(phi, c, T_celsius, m)
             for phi, c, m in zip(cells_hi, c_maps, masks)]
    if plot:
        validate_high_bias_vi(Vi_hi, hi_V, cells_hi, nrows, ncols)

    # -- Step 6: U(r) maps ------------------------------------------------
    U_maps = [calculate_local_voltage_map(phi, c, T_celsius, m)
              for phi, c, m in zip(cells_hi, c_maps, masks)]
    if plot:
        avg_U = validate_local_voltage_map(U_maps, masks, hi_V, nrows, ncols)
    else:
        avg_U = [float(np.nanmean(U[m])) for U, m in zip(U_maps, masks)]
    module_U = np.nansum(avg_U)

    # -- Step 7: J(r) maps ------------------------------------------------
    J_maps = [calculate_local_current_density_map(J0, U, T_celsius, m)
              for J0, U, m in zip(J0_maps, U_maps, masks)]
    if plot:
        int_J = validate_local_current_density_map(J_maps, masks, hi_I, nrows, ncols)
    else:
        int_J = [float(np.nansum(J[m])) for J, m in zip(J_maps, masks)]
    module_intJ = float(np.nanmedian(int_J))

    # -- Step 8: Rs(r) maps -----------------------------------------------
    Rs_maps = [calculate_series_resistance_map(Vi, U, J, m)
               for Vi, U, J, m in zip(Vi_hi, U_maps, J_maps, masks)]
    eff_Rs  = [calculate_cell_Rs_ohm(Vi, U, J, m)
               for Vi, U, J, m in zip(Vi_hi, U_maps, J_maps, masks)]
    if plot:
        total_Rs, Rs_interconnect = validate_series_resistance_map(
            Rs_maps, eff_Rs, Vi_hi, hi_V, hi_I, nrows, ncols)
    else:
        Rs_interconnect = (hi_V - np.sum(Vi_hi)) / hi_I
        total_Rs = np.nansum(eff_Rs) + Rs_interconnect

    # -- IV fit -----------------------------------------------------------
    IV_path = os.path.join(source_dir, "IV", mod_row["IVPath"].values[0])
    df_IV   = pd.read_csv(IV_path)
    V_meas  = df_IV["V"].values; I_meas = df_IV["I"].values

    Isc = mod_row['Isc_(A)'].values[0]; Voc = mod_row['Voc_(V)'].values[0]
    san = fit_sandia_simple(voltage=V_meas, current=I_meas, v_oc=Voc, i_sc=Isc)
    JL, J0_san, Rs_san, Rsh_san, nNsVth_san = san

    fit_fn = fit_sdm_fixed_n(V_meas, I_meas, N, T_celsius, n_fixed,
                              IL0=JL, I00=J0_san, Rs0=Rs_san, Rsh0=Rsh_san)
    
    # -- PySpice simulation -> SDM fit for effective module parameters -----
    spice_fit  = None
    spice_maps = None
    I_spice    = None
    V_spice    = None
    if _PYSPICE_AVAILABLE and run_pyspice:
        spice_maps = build_pyspice_maps(
            eff_J0          = eff_J0,
            eff_Rs          = eff_Rs,
            Rs_interconnect = Rs_interconnect,
            fit_fn          = fit_fn,
            num_cells       = N,
            nrows           = nrows,
            ncols           = ncols,
        )
        (_, _, _, _, _,
         I_spice, V_spice) = sim_module_oneD(
            rows      = spice_maps['portrait_rows'],
            cols      = spice_maps['portrait_cols'],
            rsmap     = spice_maps['rsmap'],
            qmap      = spice_maps['qmap'],
            jmap      = spice_maps['jmap'],
            rshmap    = spice_maps['rshmap'],
            bypass    = False,
            return_IV = True,
        )
        # Fit SDM to PySpice I-V -> effective module parameters
        Isc_spice, Voc_spice = extract_IscVoc(I_spice, V_spice)
        spice_fit = fit_sdm_fixed_n(V_spice, I_spice, N, T_celsius, n_fixed,
                                     Isc0=Isc_spice, Voc0=Voc_spice)

    if (plot or plot_final) and spice_fit is not None:
        validate_iv_fit(
            fit_fn, V_meas, I_meas, Isc, Voc,
            nNsVth_san, spice_fit['I0'], spice_fit['Rs'], Rsh_san, N, n_fixed)
    elif plot or plot_final:
        print("PySpice unavailable — skipping effective module IV comparison.")

    print(f"\nSummary — {mod_name}")
    print(f"  factor f           = {factor_f:.4e}")
    print(f"  cell Rs (Rajput)   = {total_Rs:.4f} Ohm")
    if spice_fit is not None:
        print(f"  module I0 (PySpice SDM) = {spice_fit['I0']:.2e} A")
        print(f"  module Rs (PySpice SDM) = {spice_fit['Rs']:.4f} Ohm")
    print(f"  Isc                = {Isc:.4f} A")
    print(f"  Voc                = {Voc:.4f} V")

    return dict(
        mod_name=mod_name, data=data,
        factor_f=factor_f,
        Vi_lo=Vi_lo, c_maps=c_maps,
        J0_maps=J0_maps, eff_J0=eff_J0,
        Vi_hi=Vi_hi,
        U_maps=U_maps, avg_U=avg_U, module_U=module_U,
        J_maps=J_maps, int_J=int_J, module_intJ=module_intJ,
        Rs_maps=Rs_maps, eff_Rs=eff_Rs, total_Rs=total_Rs,
        Rs_interconnect=Rs_interconnect,
        Isc=Isc, Voc=Voc,
        JL=JL, J0_san=J0_san, Rs_san=Rs_san, Rsh_san=Rsh_san, nNsVth_san=nNsVth_san,
        fit_fn=fit_fn,
        V_meas=V_meas, I_meas=I_meas,
        spice_fit=spice_fit, spice_maps=spice_maps,
        I_spice=I_spice, V_spice=V_spice,
    )

# Ver Fixed

In [21]:

def run_rajput_pipeline(mod_name, IBC, cell_dir, df, source_dir,
                         T_celsius=25, n_fixed=1,
                         plot=True, plot_final=True, run_pyspice=False):
    """
    Run the full Rajput EL-to-IV extraction pipeline for one module.

    Steps:
        0. Load & preprocess images
        1. Factor f
        2. Low-bias cell voltage Vi
        3. Calibration map c(r)
        4. Dark saturation current J0(r)
        5. High-bias cell voltage Vi
        6. Local voltage U(r)
        7. Local current density J(r)
        8. Series resistance Rs(r)
        +  Sandia + constrained SDM IV fit

    Args:
        mod_name:    module identifier string (e.g. "2531_36_1_08062019")
        IBC:         True if IBC cell (suppresses busbar detection)
        cell_dir:    path to EL cell image directory
        df:          metadata DataFrame
        source_dir:  dataset root directory
        T_celsius:   measurement temperature [degC]
        n_fixed:     ideality factor for constrained SDM fit
        plot:        if True, produce all intermediate step diagnostic plots
        plot_final:  if True, produce the final measured vs diode-fit vs
                     PySpice SDM IV comparison plot

    Returns:
        dict with all intermediate and final results.
    """
    print(f"\n{'='*60}")
    print(f"  Module: {mod_name}  |  IBC={IBC}  |  T={T_celsius}degC")
    print(f"{'='*60}")

    # -- Step 0: load data ------------------------------------------------
    data      = load_module_data(mod_name, IBC, cell_dir, df)
    cells_hi  = data['cells_hi'];  cells_lo  = data['cells_lo']
    masks     = data['cell_masks']; mod_row   = data['mod_row']
    nrows     = data['nrows'];      ncols     = data['ncols']
    N         = data['num_cells']
    lo_I      = data['lo_applied_I']; lo_V = data['lo_applied_V']
    hi_I      = data['hi_applied_I']; hi_V = data['hi_applied_V']

    if plot:
        validate_loaded_data(data)
        validate_busbar_masks(masks, cells_lo, cells_hi, nrows, ncols)

    # -- Step 1: factor f -------------------------------------------------
    factor_f = calculate_constant_f(cells_lo, masks, lo_I, lo_V, T_celsius, N)
    if plot:
        validate_constant_f(factor_f)

    # -- Step 2: low-bias Vi ----------------------------------------------
    Vi_lo = [calculate_cell_voltage_Vi(phi, m, lo_I, factor_f, T_celsius)
             for phi, m in zip(cells_lo, masks)]
    if plot:
        validate_cell_voltage_Vi(Vi_lo, lo_V, cells_lo, nrows, ncols)

    # -- Step 3: calibration map c(r) -------------------------------------
    c_maps = [calculate_calibration_constant_map(phi, Vi, m, T_celsius)
              for phi, Vi, m in zip(cells_lo, Vi_lo, masks)]
    if plot:
        validate_calibration_map(c_maps, cells_lo, masks, Vi_lo, nrows, ncols)

    # -- Step 4: J0(r) maps -----------------------------------------------
    J0_maps    = [calculate_dark_saturation_current_map(m, c, factor_f)
                  for m, c in zip(masks, c_maps)]
    eff_J0     = [float(np.nansum(J[m])) for J, m in zip(J0_maps, masks)]
    if plot:
        validate_j0_maps(J0_maps, eff_J0, nrows, ncols)

    # -- Step 5: high-bias Vi ---------------------------------------------
    Vi_hi = [calculate_high_bias_cell_voltage_Vi(phi, c, T_celsius, m)
             for phi, c, m in zip(cells_hi, c_maps, masks)]
    if plot:
        validate_high_bias_vi(Vi_hi, hi_V, cells_hi, nrows, ncols)

    # -- Step 6: U(r) maps ------------------------------------------------
    U_maps = [calculate_local_voltage_map(phi, c, T_celsius, m)
              for phi, c, m in zip(cells_hi, c_maps, masks)]
    if plot:
        avg_U = validate_local_voltage_map(U_maps, masks, hi_V, nrows, ncols)
    else:
        avg_U = [float(np.nanmean(U[m])) for U, m in zip(U_maps, masks)]
    module_U = np.nansum(avg_U)

    # -- Step 7: J(r) maps ------------------------------------------------
    J_maps = [calculate_local_current_density_map(J0, U, T_celsius, m)
              for J0, U, m in zip(J0_maps, U_maps, masks)]
    if plot:
        int_J = validate_local_current_density_map(J_maps, masks, hi_I, nrows, ncols)
    else:
        int_J = [float(np.nansum(J[m])) for J, m in zip(J_maps, masks)]
    module_intJ = float(np.nanmedian(int_J))

    # -- Step 8: Rs(r) maps -----------------------------------------------
    
    # -- Step 8: Rs(r) maps -----------------------------------------------
    print("Step 8a: calculating Rs maps...", flush=True)

    Rs_maps = [
        calculate_series_resistance_map(Vi, U, J, m)
        for Vi, U, J, m in zip(Vi_hi, U_maps, J_maps, masks)
    ]

    eff_Rs = [
        calculate_cell_Rs_ohm(Vi, U, J, m)
        for Vi, U, J, m in zip(Vi_hi, U_maps, J_maps, masks)
    ]

    print("Step 8b: Rs maps calculated", flush=True)

    if plot:
        total_Rs, Rs_interconnect = validate_series_resistance_map(
            Rs_maps, eff_Rs, Vi_hi, hi_V, hi_I, nrows, ncols
        )
        
    else:
        Rs_interconnect = (hi_V - np.sum(Vi_hi)) / hi_I
        total_Rs = np.nansum(eff_Rs) + Rs_interconnect

    print("Step 8c: Rs validation finished", flush=True)

# -- IV fit -----------------------------------------------------------
    print("Step 9a: preparing IV path...", flush=True)

    iv_relative = mod_row["IVPath"].values[0]
    print("IVPath from dataframe:", repr(iv_relative), flush=True)

    IV_path = os.path.join(source_dir, "IV", iv_relative)
    print("Full IV path:", IV_path, flush=True)
    print("IV file exists:", os.path.isfile(IV_path), flush=True)

    print("Step 9b: reading IV file...", flush=True)
    df_IV = pd.read_csv(IV_path)

    print("IV columns:", df_IV.columns.tolist(), flush=True)
    print("IV shape:", df_IV.shape, flush=True)

    V_meas = df_IV["V"].to_numpy(dtype=float)
    I_meas = df_IV["I"].to_numpy(dtype=float)

    print("Step 9c: IV data loaded", flush=True)
    print("V finite:", np.isfinite(V_meas).sum(), "/", len(V_meas), flush=True)
    print("I finite:", np.isfinite(I_meas).sum(), "/", len(I_meas), flush=True)

    Isc = float(mod_row["Isc_(A)"].values[0])
    Voc = float(mod_row["Voc_(V)"].values[0])

    print(f"Isc={Isc}, Voc={Voc}", flush=True)
    print("Step 9d: starting Sandia fit...", flush=True)

    san = fit_sandia_simple(
        voltage=V_meas,
        current=I_meas,
        v_oc=Voc,
        i_sc=Isc,
    )

    print("Step 9e: Sandia fit finished", flush=True)
    print("Sandia result:", san, flush=True)

    JL, J0_san, Rs_san, Rsh_san, nNsVth_san = san

    print("Step 9f: starting constrained SDM fit...", flush=True)

    fit_fn = fit_sdm_fixed_n(
        V_meas,
        I_meas,
        N,
        T_celsius,
        n_fixed,
        IL0=JL,
        I00=J0_san,
        Rs0=Rs_san,
        Rsh0=Rsh_san,
    )

    print("Step 9g: constrained SDM fit finished", flush=True)
    
    # -- PySpice simulation -> SDM fit for effective module parameters -----
    spice_fit  = None
    spice_maps = None
    I_spice    = None
    V_spice    = None
    if _PYSPICE_AVAILABLE and run_pyspice:
        spice_maps = build_pyspice_maps(
            eff_J0          = eff_J0,
            eff_Rs          = eff_Rs,
            Rs_interconnect = Rs_interconnect,
            fit_fn          = fit_fn,
            num_cells       = N,
            nrows           = nrows,
            ncols           = ncols,
        )
        (_, _, _, _, _,
         I_spice, V_spice) = sim_module_oneD(
            rows      = spice_maps['portrait_rows'],
            cols      = spice_maps['portrait_cols'],
            rsmap     = spice_maps['rsmap'],
            qmap      = spice_maps['qmap'],
            jmap      = spice_maps['jmap'],
            rshmap    = spice_maps['rshmap'],
            bypass    = False,
            return_IV = True,
        )
        # Fit SDM to PySpice I-V -> effective module parameters
        Isc_spice, Voc_spice = extract_IscVoc(I_spice, V_spice)
        spice_fit = fit_sdm_fixed_n(V_spice, I_spice, N, T_celsius, n_fixed,
                                     Isc0=Isc_spice, Voc0=Voc_spice)

    if (plot or plot_final) and spice_fit is not None:
        validate_iv_fit(
            fit_fn, V_meas, I_meas, Isc, Voc,
            nNsVth_san, spice_fit['I0'], spice_fit['Rs'], Rsh_san, N, n_fixed)
    elif plot or plot_final:
        print("PySpice unavailable — skipping effective module IV comparison.")

    print(f"\nSummary — {mod_name}")
    print(f"  factor f           = {factor_f:.4e}")
    print(f"  cell Rs (Rajput)   = {total_Rs:.4f} Ohm")
    if spice_fit is not None:
        print(f"  module I0 (PySpice SDM) = {spice_fit['I0']:.2e} A")
        print(f"  module Rs (PySpice SDM) = {spice_fit['Rs']:.4f} Ohm")
    print(f"  Isc                = {Isc:.4f} A")
    print(f"  Voc                = {Voc:.4f} V")

    return dict(
        mod_name=mod_name, data=data,
        factor_f=factor_f,
        Vi_lo=Vi_lo, c_maps=c_maps,
        J0_maps=J0_maps, eff_J0=eff_J0,
        Vi_hi=Vi_hi,
        U_maps=U_maps, avg_U=avg_U, module_U=module_U,
        J_maps=J_maps, int_J=int_J, module_intJ=module_intJ,
        Rs_maps=Rs_maps, eff_Rs=eff_Rs, total_Rs=total_Rs,
        Rs_interconnect=Rs_interconnect,
        Isc=Isc, Voc=Voc,
        JL=JL, J0_san=J0_san, Rs_san=Rs_san, Rsh_san=Rsh_san, nNsVth_san=nNsVth_san,
        fit_fn=fit_fn,
        V_meas=V_meas, I_meas=I_meas,
        spice_fit=spice_fit, spice_maps=spice_maps,
        I_spice=I_spice, V_spice=V_spice,
    )

## Run on your modules
EDIT: the module names below (`2531_36_1_08062019`, etc.) are Zubair's
example modules from his own subset of the dataset. Replace with your own
`module_code` values from `AnonDB_zub_60.csv` / your working CSV. Keep
`plot=True` only for the module(s) you want full diagnostic plots for —
it's slow; use `plot=False, plot_final=True` for a quick summary + final
IV comparison only.

In [22]:

res1 = run_rajput_pipeline(
    mod_name   = "2530_36_1_08062019",  # EDIT: replace with your module code
    IBC        = True,
    cell_dir   = cell_dir,
    df         = df,
    source_dir = source_dir,
    T_celsius  = 25,
    n_fixed    = 1,
    plot       = True,
    plot_final = False,
    run_pyspice = False
)


  Module: 2530_36_1_08062019  |  IBC=True  |  T=25degC


FileNotFoundError: Missing low-bias folder: C:\Users\Ghozy Abror\OneDrive - Institut Teknologi Bandung\Karirku\UNSW\Thesis\Coding\EL_cell_low_res_2\2530_36_1_08062019_20

# Batch Run

In [23]:
def calculate_pyspice_mpp(I_spice, V_spice):
    """
    Calculate Pmp, Vmp, and Imp from a PySpice IV curve.

    Handles both possible PySpice current sign conventions.
    """

    I = np.asarray(I_spice, dtype=float).ravel()
    V = np.asarray(V_spice, dtype=float).ravel()

    valid = (
        np.isfinite(I)
        & np.isfinite(V)
        & (V >= 0)
    )

    I = I[valid]
    V = V[valid]

    if len(I) == 0:
        return np.nan, np.nan, np.nan

    power_positive = V * I
    power_negative = -V * I

    if np.nanmax(power_positive) >= np.nanmax(power_negative):
        power = power_positive
        current_for_power = I
    else:
        power = power_negative
        current_for_power = -I

    valid_power = np.isfinite(power) & (power >= 0)

    if not np.any(valid_power):
        return np.nan, np.nan, np.nan

    valid_indices = np.where(valid_power)[0]
    local_index = np.nanargmax(power[valid_power])
    mpp_index = valid_indices[local_index]

    return (
        float(power[mpp_index]),
        float(V[mpp_index]),
        float(current_for_power[mpp_index]),
    )

In [26]:
import os
import time
import traceback
import numpy as np
import pandas as pd


def run_rajput_all_modules(
    excel_path,
    cell_dir,
    source_dir,
    output_csv="rajput_all_modules_results.csv",
    sheet_name=0,
    module_column="module",
    ibc_column=None,
    default_ibc=False,
    T_celsius=25,
    n_fixed=1,
    plot=False,
    plot_final=False,
    run_pyspice=True,
    max_modules=None,
    selected_modules=None,
    resume=True,
):
    """
    Run run_rajput_pipeline() for multiple modules listed in AnonDB.xlsx.

    Parameters
    ----------
    excel_path : str
        Path to AnonDB.xlsx.

    cell_dir : str
        Directory containing the cell-level EL images.

    source_dir : str
        Dataset root directory used by run_rajput_pipeline().

    output_csv : str
        CSV file where summary results are saved.

    sheet_name : str or int
        Excel sheet name or index.

    module_column : str
        Column containing module identifiers such as:
        2531_36_1_08062019

    ibc_column : str or None
        Column indicating whether each module is IBC.
        If None, default_ibc is used for every module.

    default_ibc : bool
        Default IBC value when ibc_column is not supplied.

    selected_modules : list or None
        Optional list of specific module names to process.

    max_modules : int or None
        Limit the number of modules, useful for testing.

    resume : bool
        Skip modules already present in output_csv.

    Returns
    -------
    pandas.DataFrame
        Batch summary results.
    """

    print("=" * 70)
    print("LOADING MODULE DATABASE")
    print("=" * 70)

    df = pd.read_excel(excel_path, sheet_name=sheet_name)

    print("Excel file:", excel_path)
    print("Data shape:", df.shape)
    print("Available columns:")
    print(df.columns.tolist())

    if module_column not in df.columns:
        raise KeyError(
            f"Module column {module_column!r} was not found.\n"
            f"Available columns:\n{df.columns.tolist()}"
        )

    # Keep only rows with valid module names
    work_df = df.copy()

    work_df[module_column] = (
        work_df[module_column]
        .astype(str)
        .str.strip()
        .str.replace(".0", "", regex=False)
    )

    work_df = work_df[
        work_df[module_column].notna()
        & (work_df[module_column] != "")
        & (work_df[module_column].str.lower() != "nan")
    ].copy()

    # Remove duplicate module entries
    work_df = work_df.drop_duplicates(subset=[module_column])

    # Optional manual module selection
    if selected_modules is not None:
        selected_modules = {str(x).strip() for x in selected_modules}

        work_df = work_df[
            work_df[module_column].isin(selected_modules)
        ].copy()

    # Resume from existing output
    completed_modules = set()

    if resume and os.path.isfile(output_csv):
        previous_df = pd.read_csv(output_csv)

        if "mod_name" in previous_df.columns:
            completed_modules = set(
                previous_df.loc[
                    previous_df["status"] == "ok",
                    "mod_name"
                ].astype(str)
            )

            print(
                f"Resume enabled: {len(completed_modules)} "
                "successfully completed modules found."
            )

            work_df = work_df[
                ~work_df[module_column].isin(completed_modules)
            ].copy()

    if max_modules is not None:
        work_df = work_df.head(max_modules)

    total_modules = len(work_df)

    print(f"Modules to process: {total_modules}")

    if total_modules == 0:
        print("No new modules need to be processed.")

        if os.path.isfile(output_csv):
            return pd.read_csv(output_csv)

        return pd.DataFrame()

    batch_results = []

    for batch_index, (_, row) in enumerate(work_df.iterrows(), start=1):

        mod_name = str(row[module_column]).strip()

        # Determine IBC status
        if ibc_column is None:
            IBC = bool(default_ibc)
        else:
            if ibc_column not in work_df.columns:
                raise KeyError(
                    f"IBC column {ibc_column!r} was not found."
                )

            ibc_value = row[ibc_column]

            if pd.isna(ibc_value):
                IBC = bool(default_ibc)

            elif isinstance(ibc_value, str):
                IBC = ibc_value.strip().lower() in {
                    "true",
                    "yes",
                    "y",
                    "1",
                    "ibc",
                }

            else:
                IBC = bool(ibc_value)

        print("\n" + "#" * 70)
        print(
            f"[{batch_index}/{total_modules}] "
            f"Processing {mod_name} | IBC={IBC}"
        )
        print("#" * 70)

        start_time = time.time()

        try:
            result = run_rajput_pipeline(
                mod_name=mod_name,
                IBC=IBC,
                cell_dir=cell_dir,
                df=df,
                source_dir=source_dir,
                T_celsius=T_celsius,
                n_fixed=n_fixed,
                plot=plot,
                plot_final=plot_final,
                run_pyspice=run_pyspice,
            )

            elapsed_seconds = time.time() - start_time

            fit_fn = result.get("fit_fn") or {}
            spice_fit = result.get("spice_fit") or {}

            I_spice = result.get("I_spice")
            V_spice = result.get("V_spice")

            # Calculate PySpice MPP if IV arrays exist
            Pmp_spice = np.nan
            Vmp_spice = np.nan
            Imp_spice = np.nan

            if I_spice is not None and V_spice is not None:
                (
                    Pmp_spice,
                    Vmp_spice,
                    Imp_spice,
                ) = calculate_pyspice_mpp(
                    I_spice,
                    V_spice,
                )

            result_row = {
                "mod_name": mod_name,
                "IBC": IBC,
                "status": "ok",
                "error": "",
                "elapsed_seconds": elapsed_seconds,

                # Measurement
                "Isc_measured_A": result.get("Isc", np.nan),
                "Voc_measured_V": result.get("Voc", np.nan),

                # Rajput
                "factor_f": result.get("factor_f", np.nan),
                "module_U_V": result.get("module_U", np.nan),
                "module_integrated_J_A": result.get(
                    "module_intJ",
                    np.nan,
                ),
                "Rs_cell_sum_Ohm": np.nansum(
                    result.get("eff_Rs", [])
                ),
                "Rs_interconnect_Ohm": result.get(
                    "Rs_interconnect",
                    np.nan,
                ),
                "Rs_total_Rajput_Ohm": result.get(
                    "total_Rs",
                    np.nan,
                ),
                "J0_cell_sum_A": np.nansum(
                    result.get("eff_J0", [])
                ),

                # Sandia
                "IL_sandia_A": result.get("JL", np.nan),
                "I0_sandia_A": result.get("J0_san", np.nan),
                "Rs_sandia_Ohm": result.get("Rs_san", np.nan),
                "Rsh_sandia_Ohm": result.get("Rsh_san", np.nan),
                "nNsVth_sandia_V": result.get(
                    "nNsVth_san",
                    np.nan,
                ),

                # Constrained measured-IV SDM fit
                "IL_fixed_n_A": fit_fn.get("IL", np.nan),
                "I0_fixed_n_A": fit_fn.get("I0", np.nan),
                "Rs_fixed_n_Ohm": fit_fn.get("Rs", np.nan),
                "Rsh_fixed_n_Ohm": fit_fn.get("Rsh", np.nan),

                # PySpice effective SDM fit
                "IL_pyspice_fit_A": spice_fit.get(
                    "IL",
                    np.nan,
                ),
                "I0_pyspice_fit_A": spice_fit.get(
                    "I0",
                    np.nan,
                ),
                "Rs_pyspice_fit_Ohm": spice_fit.get(
                    "Rs",
                    np.nan,
                ),
                "Rsh_pyspice_fit_Ohm": spice_fit.get(
                    "Rsh",
                    np.nan,
                ),

                # PySpice maximum power point
                "Pmp_pyspice_W": Pmp_spice,
                "Vmp_pyspice_V": Vmp_spice,
                "Imp_pyspice_A": Imp_spice,
            }

            print(
                f"SUCCESS: {mod_name} "
                f"({elapsed_seconds:.1f} seconds)"
            )

        except Exception as exc:
            elapsed_seconds = time.time() - start_time

            result_row = {
                "mod_name": mod_name,
                "IBC": IBC,
                "status": "failed",
                "error": str(exc),
                "elapsed_seconds": elapsed_seconds,
            }

            print(f"FAILED: {mod_name}")
            print(f"Error: {exc}")
            traceback.print_exc()

        batch_results.append(result_row)

        # Save immediately after every module
        current_df = pd.DataFrame(batch_results)

        if resume and os.path.isfile(output_csv):
            previous_df = pd.read_csv(output_csv)

            combined_df = pd.concat(
                [previous_df, current_df],
                ignore_index=True,
            )

            combined_df = combined_df.drop_duplicates(
                subset=["mod_name"],
                keep="last",
            )

        else:
            combined_df = current_df

        combined_df.to_csv(output_csv, index=False)

        print(f"Progress saved to: {output_csv}")

    final_df = pd.read_csv(output_csv)

    success_count = (
        final_df["status"].eq("ok").sum()
        if "status" in final_df.columns
        else 0
    )

    failed_count = (
        final_df["status"].eq("failed").sum()
        if "status" in final_df.columns
        else 0
    )

    print("\n" + "=" * 70)
    print("BATCH PROCESSING FINISHED")
    print("=" * 70)
    print(f"Successful modules : {success_count}")
    print(f"Failed modules     : {failed_count}")
    print(f"Results file       : {output_csv}")

    return final_df

In [28]:
from pathlib import Path
import pandas as pd


def get_available_modules(cell_dir):
    """
    Detect modules that have both high- and low-bias cell folders.

    Expected folder naming:
        module_name_80
        module_name_20
    """

    cell_dir = Path(cell_dir)

    high_modules = {
        folder.name[:-3]
        for folder in cell_dir.iterdir()
        if folder.is_dir() and folder.name.endswith("_80")
    }

    low_modules = {
        folder.name[:-3]
        for folder in cell_dir.iterdir()
        if folder.is_dir() and folder.name.endswith("_20")
    }

    available = sorted(high_modules & low_modules)

    print("High-bias folders :", len(high_modules))
    print("Low-bias folders  :", len(low_modules))
    print("Complete modules  :", len(available))

    return available

In [29]:
available_modules = get_available_modules(cell_dir)

print(available_modules[:10])

High-bias folders : 400
Low-bias folders  : 392
Complete modules  : 391
['10084_35_1_01272020', '10084_35_1_02222021', '10085_35_1_01272020', '10085_35_1_02222021', '10105_24_3_03012021', '10109_24_3_03012021', '10195_24_3_03012021', '10196_24_3_03012021', '10209_15_2_01142020', '10319_34_5_01202020']


In [30]:
results_test = run_rajput_all_modules(
    excel_path=r"AnonDB_test_with_Num_Cells_filtered.xlsx",
    cell_dir=cell_dir,
    source_dir=source_dir,
    output_csv="rajput_filtered_fresh2.csv",
    module_column="module",
    ibc_column="IBC",
    selected_modules=available_modules,
    default_ibc=False,
    T_celsius=25,
    n_fixed=1,
    plot=False,
    plot_final=False,
    run_pyspice=True,
    max_modules=None,
    resume=False,
)

LOADING MODULE DATABASE
Excel file: AnonDB_test_with_Num_Cells_filtered.xlsx
Data shape: (45, 67)
Available columns:
['Unnamed: 0', 'Mod_ID', 'Confidential', 'Make', 'Model', 'Interconnect_Tech', 'Module_Area_(cm2)', 'Junction_Box_Type', 'Cell_Wafer_Type', 'Cell_Tech', 'Cell_Area_(cm2)', 'Total_Exposure', 'Nameplate_Isc_(A)', 'Nameplate_Voc_(V)', 'Nameplate_Imp_(A)', 'Nameplate_Vmp_(V)', 'Nameplate_Pmp_(W)', 'Isc_(A)', 'Voc_(V)', 'Imp_(A)', 'Vmp_(V)', 'Pmp_(W)', 'FF_(percent)', 'Measured_Temperature_(C)', 'Temp_Measurement_Method', 'Voltage_Temperature_Coefficient_(mV/C)', 'Simulator_Make', 'Simulator_Model', 'IV_Date', 'IV_Time', 'IV_Lab_Location', 'Camera_Make', 'Camera_Model', 'Detector_Type', 'Image_Resolution_(MP)', 'Longpass_Filter_Wavelength_(nm)', 'High_Applied_Current_(A)', 'High_Applied_Voltage_(V)', 'High_Sensor_Exposure_Time_(s)', 'Low_Applied_Current_(A)', 'Low_Applied_Voltage_(V)', 'Low_Sensor_Exposure_Time_(s)', 'ISO', 'Aperture', 'High_Temperature_(C)', 'Low_Temperature

Traceback (most recent call last):
  File "C:\Users\Ghozy Abror\AppData\Local\Temp\ipykernel_16336\2293554092.py", line 196, in run_rajput_all_modules
    result = run_rajput_pipeline(
  File "C:\Users\Ghozy Abror\AppData\Local\Temp\ipykernel_16336\1624167876.py", line 138, in run_rajput_pipeline
    df_IV = pd.read_csv(IV_path)
  File "c:\Users\Ghozy Abror\miniconda3\envs\pyspice_test\lib\site-packages\pandas\io\parsers\readers.py", line 948, in read_csv
    return _read(filepath_or_buffer, kwds)
  File "c:\Users\Ghozy Abror\miniconda3\envs\pyspice_test\lib\site-packages\pandas\io\parsers\readers.py", line 611, in _read
    parser = TextFileReader(filepath_or_buffer, **kwds)
  File "c:\Users\Ghozy Abror\miniconda3\envs\pyspice_test\lib\site-packages\pandas\io\parsers\readers.py", line 1448, in __init__
    self._engine = self._make_engine(f, self.engine)
  File "c:\Users\Ghozy Abror\miniconda3\envs\pyspice_test\lib\site-packages\pandas\io\parsers\readers.py", line 1705, in _make_engin

Step 8a: calculating Rs maps...
Step 8b: Rs maps calculated
Step 8c: Rs validation finished
Step 9a: preparing IV path...
IVPath from dataframe: './IV/10085_35_1_02112021.csv'
Full IV path: C:\Users\Ghozy Abror\OneDrive - Institut Teknologi Bandung\Karirku\UNSW\Thesis\Coding\IV\./IV/10085_35_1_02112021.csv
IV file exists: False
Step 9b: reading IV file...
FAILED: 10085_35_1_02222021
Error: [Errno 2] No such file or directory: 'C:\\Users\\Ghozy Abror\\OneDrive - Institut Teknologi Bandung\\Karirku\\UNSW\\Thesis\\Coding\\IV\\./IV/10085_35_1_02112021.csv'
Progress saved to: rajput_filtered_fresh2.csv

######################################################################
[3/45] Processing 10105_24_3_03012021 | IBC=False
######################################################################

  Module: 10105_24_3_03012021  |  IBC=False  |  T=25degC


Traceback (most recent call last):
  File "C:\Users\Ghozy Abror\AppData\Local\Temp\ipykernel_16336\2293554092.py", line 196, in run_rajput_all_modules
    result = run_rajput_pipeline(
  File "C:\Users\Ghozy Abror\AppData\Local\Temp\ipykernel_16336\1624167876.py", line 138, in run_rajput_pipeline
    df_IV = pd.read_csv(IV_path)
  File "c:\Users\Ghozy Abror\miniconda3\envs\pyspice_test\lib\site-packages\pandas\io\parsers\readers.py", line 948, in read_csv
    return _read(filepath_or_buffer, kwds)
  File "c:\Users\Ghozy Abror\miniconda3\envs\pyspice_test\lib\site-packages\pandas\io\parsers\readers.py", line 611, in _read
    parser = TextFileReader(filepath_or_buffer, **kwds)
  File "c:\Users\Ghozy Abror\miniconda3\envs\pyspice_test\lib\site-packages\pandas\io\parsers\readers.py", line 1448, in __init__
    self._engine = self._make_engine(f, self.engine)
  File "c:\Users\Ghozy Abror\miniconda3\envs\pyspice_test\lib\site-packages\pandas\io\parsers\readers.py", line 1705, in _make_engin

Step 8a: calculating Rs maps...
Step 8b: Rs maps calculated
Step 8c: Rs validation finished
Step 9a: preparing IV path...
IVPath from dataframe: './IV/10105_24_3_02122021.csv'
Full IV path: C:\Users\Ghozy Abror\OneDrive - Institut Teknologi Bandung\Karirku\UNSW\Thesis\Coding\IV\./IV/10105_24_3_02122021.csv
IV file exists: False
Step 9b: reading IV file...
FAILED: 10105_24_3_03012021
Error: [Errno 2] No such file or directory: 'C:\\Users\\Ghozy Abror\\OneDrive - Institut Teknologi Bandung\\Karirku\\UNSW\\Thesis\\Coding\\IV\\./IV/10105_24_3_02122021.csv'
Progress saved to: rajput_filtered_fresh2.csv

######################################################################
[4/45] Processing 10196_24_3_03012021 | IBC=False
######################################################################

  Module: 10196_24_3_03012021  |  IBC=False  |  T=25degC


...

Traceback (most recent call last):
  File "C:\Users\Ghozy Abror\AppData\Local\Temp\ipykernel_16336\2293554092.py", line 196, in run_rajput_all_modules
    result = run_rajput_pipeline(
  File "C:\Users\Ghozy Abror\AppData\Local\Temp\ipykernel_16336\1624167876.py", line 138, in run_rajput_pipeline
    df_IV = pd.read_csv(IV_path)
  File "c:\Users\Ghozy Abror\miniconda3\envs\pyspice_test\lib\site-packages\pandas\io\parsers\readers.py", line 948, in read_csv
    return _read(filepath_or_buffer, kwds)
  File "c:\Users\Ghozy Abror\miniconda3\envs\pyspice_test\lib\site-packages\pandas\io\parsers\readers.py", line 611, in _read
    parser = TextFileReader(filepath_or_buffer, **kwds)
  File "c:\Users\Ghozy Abror\miniconda3\envs\pyspice_test\lib\site-packages\pandas\io\parsers\readers.py", line 1448, in __init__
    self._engine = self._make_engine(f, self.engine)
  File "c:\Users\Ghozy Abror\miniconda3\envs\pyspice_test\lib\site-packages\pandas\io\parsers\readers.py", line 1705, in _make_engin

Step 8a: calculating Rs maps...
Step 8b: Rs maps calculated
Step 8c: Rs validation finished
Step 9a: preparing IV path...
IVPath from dataframe: './IV/4548_12_1_02022021.csv'
Full IV path: C:\Users\Ghozy Abror\OneDrive - Institut Teknologi Bandung\Karirku\UNSW\Thesis\Coding\IV\./IV/4548_12_1_02022021.csv
IV file exists: False
Step 9b: reading IV file...
FAILED: 4548_12_1_02082021
Error: [Errno 2] No such file or directory: 'C:\\Users\\Ghozy Abror\\OneDrive - Institut Teknologi Bandung\\Karirku\\UNSW\\Thesis\\Coding\\IV\\./IV/4548_12_1_02022021.csv'
Progress saved to: rajput_filtered_fresh2.csv

######################################################################
[45/45] Processing 4550_12_1_03232020 | IBC=False
######################################################################

  Module: 4550_12_1_03232020  |  IBC=False  |  T=25degC


Traceback (most recent call last):
  File "C:\Users\Ghozy Abror\AppData\Local\Temp\ipykernel_16336\2293554092.py", line 196, in run_rajput_all_modules
    result = run_rajput_pipeline(
  File "C:\Users\Ghozy Abror\AppData\Local\Temp\ipykernel_16336\1624167876.py", line 138, in run_rajput_pipeline
    df_IV = pd.read_csv(IV_path)
  File "c:\Users\Ghozy Abror\miniconda3\envs\pyspice_test\lib\site-packages\pandas\io\parsers\readers.py", line 948, in read_csv
    return _read(filepath_or_buffer, kwds)
  File "c:\Users\Ghozy Abror\miniconda3\envs\pyspice_test\lib\site-packages\pandas\io\parsers\readers.py", line 611, in _read
    parser = TextFileReader(filepath_or_buffer, **kwds)
  File "c:\Users\Ghozy Abror\miniconda3\envs\pyspice_test\lib\site-packages\pandas\io\parsers\readers.py", line 1448, in __init__
    self._engine = self._make_engine(f, self.engine)
  File "c:\Users\Ghozy Abror\miniconda3\envs\pyspice_test\lib\site-packages\pandas\io\parsers\readers.py", line 1705, in _make_engin

Step 8a: calculating Rs maps...
Step 8b: Rs maps calculated
Step 8c: Rs validation finished
Step 9a: preparing IV path...
IVPath from dataframe: './IV/4550_12_1_03172020.csv'
Full IV path: C:\Users\Ghozy Abror\OneDrive - Institut Teknologi Bandung\Karirku\UNSW\Thesis\Coding\IV\./IV/4550_12_1_03172020.csv
IV file exists: False
Step 9b: reading IV file...
FAILED: 4550_12_1_03232020
Error: [Errno 2] No such file or directory: 'C:\\Users\\Ghozy Abror\\OneDrive - Institut Teknologi Bandung\\Karirku\\UNSW\\Thesis\\Coding\\IV\\./IV/4550_12_1_03172020.csv'
Progress saved to: rajput_filtered_fresh2.csv

BATCH PROCESSING FINISHED
Successful modules : 0
Failed modules     : 45
Results file       : rajput_filtered_fresh2.csv


Traceback (most recent call last):
  File "C:\Users\Ghozy Abror\AppData\Local\Temp\ipykernel_16336\2293554092.py", line 196, in run_rajput_all_modules
    result = run_rajput_pipeline(
  File "C:\Users\Ghozy Abror\AppData\Local\Temp\ipykernel_16336\1624167876.py", line 138, in run_rajput_pipeline
    df_IV = pd.read_csv(IV_path)
  File "c:\Users\Ghozy Abror\miniconda3\envs\pyspice_test\lib\site-packages\pandas\io\parsers\readers.py", line 948, in read_csv
    return _read(filepath_or_buffer, kwds)
  File "c:\Users\Ghozy Abror\miniconda3\envs\pyspice_test\lib\site-packages\pandas\io\parsers\readers.py", line 611, in _read
    parser = TextFileReader(filepath_or_buffer, **kwds)
  File "c:\Users\Ghozy Abror\miniconda3\envs\pyspice_test\lib\site-packages\pandas\io\parsers\readers.py", line 1448, in __init__
    self._engine = self._make_engine(f, self.engine)
  File "c:\Users\Ghozy Abror\miniconda3\envs\pyspice_test\lib\site-packages\pandas\io\parsers\readers.py", line 1705, in _make_engin

In [36]:
def load_module_data(mod_name, IBC, cell_dir, df):
    """
    Load and preprocess cell images + metadata for one module.
    """

    mod_name = str(mod_name).strip()

    # ------------------------------------------------------------
    # 1. Cell image folders
    # ------------------------------------------------------------
    mod_hi_path = os.path.join(cell_dir, f"{mod_name}_80")
    mod_lo_path = os.path.join(cell_dir, f"{mod_name}_20")

    if not os.path.isdir(mod_hi_path):
        raise FileNotFoundError(f"Missing high-bias folder: {mod_hi_path}")

    if not os.path.isdir(mod_lo_path):
        raise FileNotFoundError(f"Missing low-bias folder: {mod_lo_path}")

    # Keep False to reproduce the original 8-bit loading behaviour.
    USE_UNCHANGED_READ = False

    read_flag = (
        cv2.IMREAD_UNCHANGED
        if USE_UNCHANGED_READ
        else cv2.IMREAD_GRAYSCALE
    )

    valid_extensions = (".tif", ".tiff")

    hi_files = sorted(
        f for f in os.listdir(mod_hi_path)
        if f.lower().endswith(valid_extensions)
    )

    lo_files = sorted(
        f for f in os.listdir(mod_lo_path)
        if f.lower().endswith(valid_extensions)
    )

    cells_hi = [
        cv2.imread(os.path.join(mod_hi_path, f), read_flag)
        for f in hi_files
    ]

    cells_lo = [
        cv2.imread(os.path.join(mod_lo_path, f), read_flag)
        for f in lo_files
    ]

    # Check failed image reads
    bad_hi = [f for f, img in zip(hi_files, cells_hi) if img is None]
    bad_lo = [f for f, img in zip(lo_files, cells_lo) if img is None]

    if bad_hi:
        raise ValueError(
            f"Failed to read high-bias images for {mod_name}: {bad_hi[:5]}"
        )

    if bad_lo:
        raise ValueError(
            f"Failed to read low-bias images for {mod_name}: {bad_lo[:5]}"
        )

    if len(cells_hi) == 0:
        raise ValueError(f"No high-bias TIFF cells found for {mod_name}")

    if len(cells_lo) == 0:
        raise ValueError(f"No low-bias TIFF cells found for {mod_name}")

    if len(cells_hi) != len(cells_lo):
        raise ValueError(
            f"Different cell counts for {mod_name}: "
            f"high={len(cells_hi)}, low={len(cells_lo)}"
        )

    num_cells = len(cells_hi)

    if num_cells == 60:
        nrows, ncols = 6, 10
    elif num_cells == 72:
        nrows, ncols = 6, 12
    elif num_cells % 6 == 0:
        nrows, ncols = 6, num_cells // 6
    else:
        raise ValueError(
            f"Unsupported number of cells for {mod_name}: {num_cells}"
        )

    # ------------------------------------------------------------
    # 2. Metadata lookup
    # ------------------------------------------------------------
    if "module" in df.columns:
        module_values = (
            df["module"]
            .astype(str)
            .str.strip()
        )

        mod_row = df.loc[module_values == mod_name].copy()

    elif "mod_name" in df.columns:
        module_values = (
            df["mod_name"]
            .astype(str)
            .str.strip()
        )

        mod_row = df.loc[module_values == mod_name].copy()

    else:
        # Fallback: recreate module name from ELHPath
        if "ELHPath" not in df.columns:
            raise KeyError(
                "Metadata requires one of these columns: "
                "'module', 'mod_name', or 'ELHPath'."
            )

        module_values = (
            df["ELHPath"]
            .astype(str)
            .str.replace("\\", "/", regex=False)
            .str.rsplit("/", n=1)
            .str[-1]
            .str.replace(r"\.tiff?$", "", regex=True, case=False)
            .str.replace(r"_80$", "", regex=True)
            .str.strip()
        )

        mod_row = df.loc[module_values == mod_name].copy()

    if mod_row.empty:
        raise ValueError(f"Metadata lookup failed for {mod_name}")

    if len(mod_row) > 1:
        print(
            f"Warning: {len(mod_row)} metadata rows found for "
            f"{mod_name}; using the first row."
        )
        mod_row = mod_row.iloc[[0]].copy()

    # ------------------------------------------------------------
    # 3. Busbar masks
    # ------------------------------------------------------------
    cell_masks = [
        create_busbar_mask(
            cell,
            IBC=IBC,
            busbar_orientation="horizontal",
        )
        for cell in cells_lo
    ]

    # ------------------------------------------------------------
    # 4. Convert image values to float32
    # ------------------------------------------------------------
    if not USE_UNCHANGED_READ:
        cells_hi = [
            cell.astype(np.float32) * (65535.0 / 255.0)
            for cell in cells_hi
        ]

        cells_lo = [
            cell.astype(np.float32) * (65535.0 / 255.0)
            for cell in cells_lo
        ]
    else:
        cells_hi = [
            cell.astype(np.float32)
            for cell in cells_hi
        ]

        cells_lo = [
            cell.astype(np.float32)
            for cell in cells_lo
        ]

    # ------------------------------------------------------------
    # 5. Prevent log(0)
    # ------------------------------------------------------------
    cells_hi = [
        np.where(mask & (cell == 0), 1, cell)
        for cell, mask in zip(cells_hi, cell_masks)
    ]

    cells_lo = [
        np.where(mask & (cell == 0), 1, cell)
        for cell, mask in zip(cells_lo, cell_masks)
    ]

    # ------------------------------------------------------------
    # 6. Read and validate metadata values
    # ------------------------------------------------------------
    hi_exp = float(
        mod_row["High_Sensor_Exposure_Time_(s)"].iloc[0]
    )

    lo_exp = float(
        mod_row["Low_Sensor_Exposure_Time_(s)"].iloc[0]
    )

    if not np.isfinite(hi_exp) or hi_exp <= 0:
        raise ValueError(
            f"Invalid high exposure time for {mod_name}: {hi_exp}"
        )

    if not np.isfinite(lo_exp) or lo_exp <= 0:
        raise ValueError(
            f"Invalid low exposure time for {mod_name}: {lo_exp}"
        )

    # Normalise by exposure time
    cells_hi = [cell / hi_exp for cell in cells_hi]
    cells_lo = [cell / lo_exp for cell in cells_lo]

    return dict(
        cells_hi=cells_hi,
        cells_lo=cells_lo,
        cell_masks=cell_masks,
        mod_row=mod_row,
        nrows=nrows,
        ncols=ncols,
        num_cells=num_cells,

        lo_applied_I=float(
            mod_row["Low_Applied_Current_(A)"].iloc[0]
        ),
        lo_applied_V=float(
            mod_row["Low_Applied_Voltage_(V)"].iloc[0]
        ),
        hi_applied_I=float(
            mod_row["High_Applied_Current_(A)"].iloc[0]
        ),
        hi_applied_V=float(
            mod_row["High_Applied_Voltage_(V)"].iloc[0]
        ),

        hi_exposure_time=hi_exp,
        lo_exposure_time=lo_exp,
    )

# sudah sampai sini aja

In [23]:
spice_maps = build_pyspice_maps(
    eff_J0=res1["eff_J0"],
    eff_Rs=res1["eff_Rs"],
    Rs_interconnect=res1["Rs_interconnect"],
    fit_fn=res1["fit_fn"],
    num_cells=res1["data"]["num_cells"],
    nrows=res1["data"]["nrows"],
    ncols=res1["data"]["ncols"],
)

PySpice Rs offset per cell: 3.003 mOhm  (Rs_interconnect=0.2162 Ohm / 72 cells)


In [28]:
excel_path = r"AnonDB.xlsx"

df_check = pd.read_excel(excel_path)

print(df_check.shape)
print(df_check.columns.tolist())
df_check.head()

(621, 72)
['Column1', 'Mod_ID', 'Confidential', 'Make', 'Model', 'Interconnect_Tech', 'Module_Area_(cm2)', 'Junction_Box_Type', 'Cell_Wafer_Type', 'Cell_Tech', 'Cell_Area_(cm2)', 'rows', 'cols', 'Num_Cells', 'IBC', 'Half_Cell', 'Total_Exposure', 'Nameplate_Isc_(A)', 'Nameplate_Voc_(V)', 'Nameplate_Imp_(A)', 'Nameplate_Vmp_(V)', 'Nameplate_Pmp_(W)', 'Isc_(A)', 'Voc_(V)', 'Imp_(A)', 'Vmp_(V)', 'Pmp_(W)', 'FF_(percent)', 'Measured_Temperature_(C)', 'Temp_Measurement_Method', 'Voltage_Temperature_Coefficient_(mV/C)', 'Simulator_Make', 'Simulator_Model', 'IV_Date', 'IV_Time', 'IV_Lab_Location', 'Camera_Make', 'Camera_Model', 'Detector_Type', 'Image_Resolution_(MP)', 'Longpass_Filter_Wavelength_(nm)', 'High_Applied_Current_(A)', 'High_Applied_Voltage_(V)', 'High_Sensor_Exposure_Time_(s)', 'Low_Applied_Current_(A)', 'Applied_Current_Diff', 'Low_Applied_Voltage_(V)', 'Applied_Voltage_Diff', 'Low_Sensor_Exposure_Time_(s)', 'Sensor_Exp_Diff', 'ISO', 'Aperture', 'High_Temperature_(C)', 'Low_Tempe

,Column1,Mod_ID,Confidential,Make,Model,Interconnect_Tech,Module_Area_(cm2),Junction_Box_Type,Cell_Wafer_Type,Cell_Tech,...,IVPath,Ref,Exposure_Step,Total_Exposure_Steps,RefIVPath,RefA,RefRefA,Ref_IV_Date,mod_name,Ref_IV_Time
0,259.0,3245.0,NaN,5.0,5.0,NaN,16394.4,NaN,multi-Si module,NaN,...,./IV/3245_5_5_02012021.csv,0.0,1.0,3.0,./RefIV/3245_5_5_02012021.csv,8.8620,9.0256,7/19/2017,3245_5_5_02012021,08:46:12
1,258.0,3245.0,NaN,5.0,5.0,NaN,16394.4,NaN,multi-Si module,NaN,...,./IV/3245_5_5_08012017.csv,0.0,2.0,3.0,./RefIV/3245_5_5_08012017.csv,9.0256,9.0256,7/19/2017,3245_5_5_08012017,08:46:12
2,260.0,3245.0,NaN,5.0,5.0,NaN,16394.4,NaN,multi-Si module,NaN,...,./IV/3245_5_5_01162020.csv,0.0,0.0,3.0,./RefIV/3245_5_5_01162020.csv,8.8620,9.0256,7/19/2017,3245_5_5_01162020,08:46:12
3,256.0,3246.0,NaN,5.0,5.0,NaN,16394.4,NaN,multi-Si module,NaN,...,./IV/3246_5_5_02012021.csv,0.0,1.0,3.0,./RefIV/3246_5_5_02012021.csv,8.8620,9.0256,7/19/2017,3246_5_5_02012021,08:58:19
4,255.0,3246.0,NaN,5.0,5.0,NaN,16394.4,NaN,multi-Si module,NaN,...,./IV/3246_5_5_08012017.csv,0.0,2.0,3.0,./RefIV/3246_5_5_08012017.csv,9.0256,9.0256,7/19/2017,3246_5_5_08012017,08:58:19


In [25]:
import os

SPICE_ROOT = r"C:\Spice64_dll"
DLL_DIR = os.path.join(SPICE_ROOT, "dll-vs")
SHARE_DIR = os.path.join(SPICE_ROOT, "share", "ngspice")

if hasattr(os, "add_dll_directory"):
    _dll_handle = os.add_dll_directory(DLL_DIR)

os.environ["PATH"] = DLL_DIR + os.pathsep + os.environ.get("PATH", "")
os.environ["SPICE_LIB_DIR"] = SHARE_DIR
os.environ["SPICE_SCRIPTS"] = os.path.join(SHARE_DIR, "scripts")

from PySpice.Spice.NgSpice.Shared import NgSpiceShared

NgSpiceShared.LIBRARY_PATH = r"C:\Spice64_dll\dll-vs\ngspice{}.dll"
NgSpiceShared.NGSPICE_PATH = r"C:\Spice64_dll"

print("Before instance", flush=True)
_ng = NgSpiceShared.new_instance()
print("Instance created", flush=True)
print(_ng.exec_command("version"))

Before instance
Instance created
******
** ngspice-34 : Circuit level simulation program
** The U. C. Berkeley CAD Group
** Copyright 1985-1994, Regents of the University of California.
** Copyright 2001-2020, The ngspice team.
** Please get your ngspice manual from http://ngspice.sourceforge.net/docs.html
** Please file your bug-reports at http://ngspice.sourceforge.net/bugrep.html
** Creation Date: Jan 29 2021   16:38:37
******


In [22]:
import os
import shutil
from pathlib import Path
import PySpice

# Lokasi ngspice bawaan PySpice
source_root = (
    Path(PySpice.__file__).parent
    / "Spice"
    / "NgSpice"
    / "Spice64_dll"
)

# Lokasi baru tanpa spasi
target_root = Path(r"C:\Spice64_dll")

print("Source:", source_root)
print("Target:", target_root)

# Copy hanya jika belum ada
if not target_root.exists():
    shutil.copytree(source_root, target_root)
    print("Folder ngspice copied.")
else:
    print("Target folder already exists.")

# Patch semua file teks konfigurasi yang masih menyebut path lama
old_path_windows = str(source_root)
old_path_forward = old_path_windows.replace("\\", "/")

for file_path in target_root.rglob("*"):
    if not file_path.is_file():
        continue

    # Hanya coba file yang kemungkinan merupakan file konfigurasi/teks
    if file_path.suffix.lower() not in {
        "", ".txt", ".ini", ".conf", ".cir", ".sp", ".cmd"
    }:
        continue

    try:
        text = file_path.read_text(encoding="utf-8", errors="ignore")
    except Exception:
        continue

    new_text = (
        text
        .replace(old_path_windows, str(target_root))
        .replace(old_path_forward, str(target_root).replace("\\", "/"))
    )

    if new_text != text:
        file_path.write_text(new_text, encoding="utf-8")
        print("Patched:", file_path)

print("Preparation complete.")

Source: c:\Users\Ghozy Abror\miniconda3\envs\pyspice_test\lib\site-packages\PySpice\Spice\NgSpice\Spice64_dll
Target: C:\Spice64_dll
Target folder already exists.
Preparation complete.


In [23]:
required_files = [
    target_root / "dll-vs" / "ngspice.dll",
    target_root / "lib" / "ngspice" / "analog.cm",
    target_root / "lib" / "ngspice" / "digital.cm",
    target_root / "lib" / "ngspice" / "spice2poly.cm",
]

for path in required_files:
    print(path, "->", path.exists())

C:\Spice64_dll\dll-vs\ngspice.dll -> True
C:\Spice64_dll\lib\ngspice\analog.cm -> True
C:\Spice64_dll\lib\ngspice\digital.cm -> True
C:\Spice64_dll\lib\ngspice\spice2poly.cm -> True


In [24]:
import os

SPICE_ROOT = r"C:\Spice64_dll"
DLL_DIR = os.path.join(SPICE_ROOT, "dll-vs")
CM_DIR = os.path.join(SPICE_ROOT, "lib", "ngspice")
SHARE_DIR = os.path.join(SPICE_ROOT, "share", "ngspice")

# Dependency DLL Windows
if hasattr(os, "add_dll_directory"):
    _dll_handle = os.add_dll_directory(DLL_DIR)

os.environ["PATH"] = DLL_DIR + os.pathsep + os.environ.get("PATH", "")
os.environ["SPICE_LIB_DIR"] = SHARE_DIR
os.environ["SPICE_SCRIPTS"] = os.path.join(SHARE_DIR, "scripts")

from PySpice.Spice.NgSpice.Shared import NgSpiceShared

NgSpiceShared.LIBRARY_PATH = (
    r"C:\Spice64_dll\dll-vs\ngspice{}.dll"
)
NgSpiceShared.NGSPICE_PATH = r"C:\Spice64_dll"

print("Before instance", flush=True)

_ng = NgSpiceShared.new_instance()

print("Instance created", flush=True)
print(_ng.exec_command("version"))

Before instance


Error: Library C:\Users\Ghozy couldn't be loaded!
Error: Library Abror\miniconda3\envs\pyspice_test\lib\site-packages\PySpice\Spice\NgSpice\Spice64_dll\lib\ngspice/spice2poly.cm couldn't be loaded!
Error: Library C:\Users\Ghozy couldn't be loaded!
Error: Library Abror\miniconda3\envs\pyspice_test\lib\site-packages\PySpice\Spice\NgSpice\Spice64_dll\lib\ngspice/analog.cm couldn't be loaded!
Error: Library C:\Users\Ghozy couldn't be loaded!
Error: Library Abror\miniconda3\envs\pyspice_test\lib\site-packages\PySpice\Spice\NgSpice\Spice64_dll\lib\ngspice/digital.cm couldn't be loaded!
Error: Library C:\Users\Ghozy couldn't be loaded!
Error: Library Abror\miniconda3\envs\pyspice_test\lib\site-packages\PySpice\Spice\NgSpice\Spice64_dll\lib\ngspice/xtradev.cm couldn't be loaded!
Error: Library C:\Users\Ghozy couldn't be loaded!
Error: Library Abror\miniconda3\envs\pyspice_test\lib\site-packages\PySpice\Spice\NgSpice\Spice64_dll\lib\ngspice/xtraevt.cm couldn't be loaded!
Error: Library C:\Users

Instance created
******
** ngspice-34 : Circuit level simulation program
** The U. C. Berkeley CAD Group
** Copyright 1985-1994, Regents of the University of California.
** Copyright 2001-2020, The ngspice team.
** Please get your ngspice manual from http://ngspice.sourceforge.net/docs.html
** Please file your bug-reports at http://ngspice.sourceforge.net/bugrep.html
** Creation Date: Jan 29 2021   16:38:37
******


In [25]:
old_fragment = "Ghozy Abror"

for file_path in target_root.rglob("*"):
    if not file_path.is_file():
        continue

    try:
        text = file_path.read_text(encoding="utf-8", errors="ignore")
    except Exception:
        continue

    if old_fragment in text:
        print("Old path found in:", file_path)

        for line in text.splitlines():
            if old_fragment in line:
                print("  ", line)

Old path found in: C:\Spice64_dll\share\ngspice\scripts\spinit
    codemodel C:\Users\Ghozy Abror\miniconda3\envs\pyspice_test\lib\site-packages\PySpice\Spice\NgSpice\Spice64_dll\lib\ngspice/spice2poly.cm
    codemodel C:\Users\Ghozy Abror\miniconda3\envs\pyspice_test\lib\site-packages\PySpice\Spice\NgSpice\Spice64_dll\lib\ngspice/analog.cm
    codemodel C:\Users\Ghozy Abror\miniconda3\envs\pyspice_test\lib\site-packages\PySpice\Spice\NgSpice\Spice64_dll\lib\ngspice/digital.cm
    codemodel C:\Users\Ghozy Abror\miniconda3\envs\pyspice_test\lib\site-packages\PySpice\Spice\NgSpice\Spice64_dll\lib\ngspice/xtradev.cm
    codemodel C:\Users\Ghozy Abror\miniconda3\envs\pyspice_test\lib\site-packages\PySpice\Spice\NgSpice\Spice64_dll\lib\ngspice/xtraevt.cm
    codemodel C:\Users\Ghozy Abror\miniconda3\envs\pyspice_test\lib\site-packages\PySpice\Spice\NgSpice\Spice64_dll\lib\ngspice/table.cm


In [26]:
for file_path in target_root.rglob("spinit*"):
    if file_path.is_file():
        text = file_path.read_text(encoding="utf-8", errors="ignore")

        text = text.replace(
            str(source_root),
            str(target_root)
        )

        text = text.replace(
            str(source_root).replace("\\", "/"),
            str(target_root).replace("\\", "/")
        )

        file_path.write_text(text, encoding="utf-8")
        print("Patched spinit:", file_path)

Patched spinit: C:\Spice64_dll\share\ngspice\scripts\spinit


In [25]:
from PySpice.Spice.NgSpice.Shared import NgSpiceShared

NgSpiceShared.NGSPICE_PATH = SPICE_ROOT
NgSpiceShared.LIBRARY_PATH = (
    r"C:\Spice64_dll\dll-vs\ngspice{}.dll"
)

print("Before instance", flush=True)

_ng = NgSpiceShared.new_instance()

print("Instance created", flush=True)
print(_ng.exec_command("version"))

Before instance


Error: Library C:\Users\Ghozy couldn't be loaded!
Error: Library Abror\miniconda3\envs\pyspice_test\lib\site-packages\PySpice\Spice\NgSpice\Spice64_dll\lib\ngspice/spice2poly.cm couldn't be loaded!
Error: Library C:\Users\Ghozy couldn't be loaded!
Error: Library Abror\miniconda3\envs\pyspice_test\lib\site-packages\PySpice\Spice\NgSpice\Spice64_dll\lib\ngspice/analog.cm couldn't be loaded!
Error: Library C:\Users\Ghozy couldn't be loaded!
Error: Library Abror\miniconda3\envs\pyspice_test\lib\site-packages\PySpice\Spice\NgSpice\Spice64_dll\lib\ngspice/digital.cm couldn't be loaded!
Error: Library C:\Users\Ghozy couldn't be loaded!
Error: Library Abror\miniconda3\envs\pyspice_test\lib\site-packages\PySpice\Spice\NgSpice\Spice64_dll\lib\ngspice/xtradev.cm couldn't be loaded!
Error: Library C:\Users\Ghozy couldn't be loaded!
Error: Library Abror\miniconda3\envs\pyspice_test\lib\site-packages\PySpice\Spice\NgSpice\Spice64_dll\lib\ngspice/xtraevt.cm couldn't be loaded!
Error: Library C:\Users

Instance created
******
** ngspice-34 : Circuit level simulation program
** The U. C. Berkeley CAD Group
** Copyright 1985-1994, Regents of the University of California.
** Copyright 2001-2020, The ngspice team.
** Please get your ngspice manual from http://ngspice.sourceforge.net/docs.html
** Please file your bug-reports at http://ngspice.sourceforge.net/bugrep.html
** Creation Date: Jan 29 2021   16:38:37
******


In [ ]:
# Uncomment and extend once you've run more than one module through the pipeline.

# results  = [res1]  # EDIT: add res2, res3, ... as you run more modules
# names    = [r['mod_name'] for r in results]
# J0_vals  = [r['spice_fit']['I0'] if r['spice_fit'] else float('nan') for r in results]
# Rs_vals  = [r['spice_fit']['Rs'] if r['spice_fit'] else r['total_Rs'] for r in results]
# Isc_vals = [r['Isc'] for r in results]
# Voc_vals = [r['Voc'] for r in results]
#
# fig, axes = plt.subplots(1, 2, figsize=(16, 8))
# axes[0].bar(names, J0_vals, color='steelblue')
# axes[0].set_ylabel("Module I0 (A)"); axes[0].set_title("Effective I0 per module (PySpice SDM)")
# axes[0].set_xticklabels(names, rotation=30, ha='right')
#
# axes[1].bar(names, Rs_vals, color='darkorange')
# axes[1].set_ylabel("Module Rs (Ohm)"); axes[1].set_title("Effective Rs per module (PySpice SDM)")
# axes[1].set_xticklabels(names, rotation=30, ha='right')
#
# plt.tight_layout(); plt.show()
#
# print("\nSummary table:")
# print(f"{'Module':<30} {'I0 (A)':>12} {'Rs (Ohm)':>10} {'Isc (A)':>10} {'Voc (V)':>10}")
# for n, j, r, i, v in zip(names, J0_vals, Rs_vals, Isc_vals, Voc_vals):
#     print(f"{n:<30} {j:>12.2e} {r:>10.4f} {i:>10.4f} {v:>10.4f}")

# Error solving

In [21]:
import time
import numpy as np

IL_test = 5.99771
I0_test = 1.013678e-9
Rs_test = 0.268744
Rsh_test = 174.489
nNsVth_test = 1.84987

V_test = np.linspace(0, 48.46, 20)

print("Starting direct i_from_v test...", flush=True)
start = time.perf_counter()

I_test = i_from_v(
    V_test,
    IL_test,
    I0_test,
    Rs_test,
    Rsh_test,
    nNsVth_test,
    method="lambertw",
)

print("Finished:", time.perf_counter() - start, "seconds", flush=True)
print(I_test)

Starting direct i_from_v test...
Finished: 0.0005920000257901847 seconds
[  5.98848667   5.97389202   5.95929735   5.9447026    5.93010752
   5.91551111   5.90090949   5.8862872    5.871583     5.85655438
   5.84024101   5.81884596   5.77744215   5.65863849   5.2592541
   4.02721006   1.1806511   -3.48282042  -9.51579806 -16.43457529]


In [22]:
import inspect
print(inspect.signature(i_from_v))
print(i_from_v.__module__)

(voltage, photocurrent, saturation_current, resistance_series, resistance_shunt, nNsVth, method='lambertw')
pvlib.pvsystem


In [23]:
I_test = i_from_v(
    V_test,
    IL_test,
    I0_test,
    Rs_test,
    Rsh_test,
    nNsVth_test,
    method="newton",
)

In [24]:
fit_fn = fit_sdm_fixed_n(
    V_fit,
    I_fit,
    N,
    T_celsius,
    n_fixed,
    IL0=JL,
    I00=J0_san,
    Rs0=Rs_san,
    Rsh0=Rsh_san,
    method="brentq",
    max_nfev=20,
)

NameError: name 'V_fit' is not defined

In [26]:
print(i_from_v)
print(i_from_v.__module__)

<function i_from_v at 0x0000020CC73EAD40>
pvlib.pvsystem


In [28]:
from pvlib.pvsystem import i_from_v as pvlib_i_from_v

In [29]:
V = V_test
print(type(V), V.shape, V.dtype)
print(V[:5])
print(V[-5:])

I_m = pvlib_i_from_v(
    voltage=42.0,
    photocurrent=5.99771,
    saturation_current=1.013678e-9,
    resistance_series=0.268744,
    resistance_shunt=174.489,
    nNsVth=1.84987,
    method="lambertw",
)

print("Direct full-array test finished")

<class 'numpy.ndarray'> (20,) float64
[ 0.          2.55052632  5.10105263  7.65157895 10.20210526]
[38.25789474 40.80842105 43.35894737 45.90947368 48.46      ]
Direct full-array test finished


In [1]:
import scipy
import pvlib
import numpy

print("numpy:", numpy.__version__)
print("scipy:", scipy.__version__)
print("pvlib:", pvlib.__version__)

numpy: 1.26.4
scipy: 1.11.4
pvlib: 0.15.0


In [23]:
NgSpiceShared.new_instance()

In [24]:
from scipy.special import lambertw

print("Before Lambert W", flush=True)
result = lambertw(1.0)
print("After Lambert W:", result, flush=True)

Before Lambert W
After Lambert W: (0.5671432904097838+0j)


In [25]:
from pvlib.pvsystem import i_from_v

print("Before pvlib i_from_v", flush=True)

result = i_from_v(
    voltage=42.0,
    photocurrent=5.99771,
    saturation_current=1.013678e-9,
    resistance_series=0.268744,
    resistance_shunt=174.489,
    nNsVth=1.84987,
    method="lambertw",
)

print("After pvlib i_from_v:", result, flush=True)

Before pvlib i_from_v
After pvlib i_from_v: -0.792195628821764


In [26]:
_ng = NgSpiceShared.new_instance()

In [27]:
def initialise_ngspice():
    from PySpice.Spice.NgSpice.Shared import NgSpiceShared

    NgSpiceShared.LIBRARY_PATH = (
        r"C:\ngspice_dll\Spice64_dll\dll-vs\ngspice{}.dll"
    )
    NgSpiceShared.NGSPICE_PATH = (
        r"C:\ngspice_dll\Spice64_dll"
    )

    ng = NgSpiceShared.new_instance()
    print("ngspice loaded", flush=True)
    return ng

In [28]:
print(_ng.exec_command("version"))

******
** ngspice-46 : Circuit level simulation program
** Compiled with KLU Direct Linear Solver
** The U. C. Berkeley CAD Group
** Copyright 1985-1994, Regents of the University of California.
** Copyright 2001-2025, The ngspice team.
** Please get your ngspice manual from https://ngspice.sourceforge.io/docs.html
** Please file your bug-reports at https://ngspice.sourceforge.io/bugrep.html
** Creation Date: Mar 29 2026   14:53:14
******


In [29]:
from scipy.special import lambertw

print("Before lambertw", flush=True)
print(lambertw(1.0))
print("After lambertw", flush=True)

Before lambertw
(0.5671432904097838+0j)
After lambertw


In [30]:
# 1. Jangan import atau initialise PySpice dulu
from scipy.special import lambertw
print(lambertw(1.0))

# 2. Test pvlib
from pvlib.pvsystem import i_from_v
print(i_from_v(
    voltage=42.0,
    photocurrent=5.99771,
    saturation_current=1.013678e-9,
    resistance_series=0.268744,
    resistance_shunt=174.489,
    nNsVth=1.84987,
    method="lambertw",
))

# 3. Baru initialise ngspice

(0.5671432904097838+0j)
-0.792195628821764
